# Classical AI Reasoning Agent
**Natural-language knowledge base → internal representation → classical reasoning → True / False**

This notebook contains the complete implementation of the reasoning system, its evaluation on the public
datasets (RuleTaker, CounterBench, FOLIO), reasoning traces, and the code that writes `submission.csv`.

| Step | What happens |
|---|---|
| Parsing | Rule-based semantic parsers turn English into literals, Horn rules, structural causal equations or first-order formulas. No machine-learning model and no generative AI is used anywhere. |
| Reasoning | Forward chaining with unification (stratified negation-as-failure), backward chaining, possible-world search for counterfactuals (abduction → action → prediction), and first-order resolution refutation with set-of-support and uniform-cost clause selection. |
| Output | `True` when the statement is entailed by the knowledge base, otherwise `False`. |

**How to run (Google Colab or local Jupyter):** `Runtime → Run all` (Colab) or `Kernel → Restart & Run All` (Jupyter).
The first cells install the libraries and log in to Hugging Face; the datasets are then downloaded automatically.

*Colab, one-time setup for FOLIO (gated dataset):* accept the terms on https://huggingface.co/datasets/yale-nlp/FOLIO,
create a read token at https://huggingface.co/settings/tokens, and add it in Colab under the key icon (Secrets) with the
name `HF_TOKEN` and notebook access switched on.
FOLIO (`yale-nlp/FOLIO`) is gated: accept its terms on Hugging Face and run `huggingface-cli login` once, or put
`FOLIO_train.jsonl` / `FOLIO_validation.jsonl` next to the notebook. If a private test file is placed next to the notebook as
`test.csv` (columns `id, context, statement`), predictions for it are written to `submission.csv` (columns `id, flag`).

In [1]:
# Setup: install libraries (needed on Colab / fresh environments; quick no-op if already installed)
import sys, subprocess
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'huggingface_hub', 'pandas', 'reportlab'],
                   capture_output=True, text=True)
print('libraries ready' if r.returncode == 0 else r.stderr[-500:])

libraries ready


In [2]:
# Hugging Face login (only needed for the gated yale-nlp/FOLIO dataset).
# Reads the Colab secret HF_TOKEN (or the HF_TOKEN environment variable) and checks it.
# A missing or invalid token never stops the notebook: public datasets download without it,
# and FOLIO then falls back to local files.
import os
HF_TOKEN = None
token = os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    token = token or userdata.get('HF_TOKEN')
except Exception:
    pass
if token:
    token = token.strip()
    try:
        from huggingface_hub import whoami
        user = whoami(token=token).get('name', '?')
        HF_TOKEN = token
        os.environ['HF_TOKEN'] = token
        print(f'Hugging Face token OK (user: {user}).')
    except Exception as e:
        os.environ.pop('HF_TOKEN', None)
        if '401' in str(e) or 'nvalid' in str(e):
            print('Hugging Face rejected the HF_TOKEN secret (invalid, incomplete or revoked token). Create a new '
                  'Read token at huggingface.co/settings/tokens, paste it into the Colab secret, then '
                  'Runtime -> Restart session and run again. Continuing without a token.')
        else:
            print(f'Could not verify the Hugging Face token ({type(e).__name__}: network problem?). Continuing without it.')
else:
    print('No HF_TOKEN found. RuleTaker and CounterBench still download; for FOLIO either add the token '
          'or place FOLIO_train.jsonl / FOLIO_validation.jsonl next to this notebook.')
# Explicit token for every download: the valid token, or False so that an invalid
# token stored in Colab is never sent with requests for public datasets.
HUB_TOKEN = HF_TOKEN or False

No HF_TOKEN found. RuleTaker and CounterBench still download; for FOLIO either add the token or place FOLIO_train.jsonl / FOLIO_validation.jsonl next to this notebook.


In [3]:
# Configuration ------------------------------------------------------------------
TEST_PATH        = 'test.csv'        # private test file (csv / json / jsonl), if provided
SUBMISSION_PATH  = 'submission.csv'
LABEL_COLUMN     = 'flag'            # name of the prediction column in submission.csv
RULETAKER_SPLITS = ['train', 'test', 'ood_test']      # jise/ruletaker
FOLIO_SPLITS     = ['train', 'validation']            # yale-nlp/FOLIO (gated: accept terms + `huggingface-cli login`)
FOLIO_LOCAL      = {'train': 'FOLIO_train.jsonl', 'validation': 'FOLIO_validation.jsonl'}  # used if the download fails

import warnings
warnings.filterwarnings('ignore')

## 1. Reasoning engine
Everything below is written from scratch in plain Python (standard library only).

In [4]:
# =============================================================================
#  Classical Reasoning Agent
#  Natural language KB -> symbolic representation -> classical inference -> T/F
#
#  Engines (all implemented from scratch, no ML / no generative AI):
#    * Forward chaining over Horn-style rules with unification and stratified
#      negation-as-failure (closed-world, RuleTaker) or classical negation
#      (open-world, ProofWriter style).
#    * Backward chaining (goal-directed proof search, used for proof traces).
#    * Possible-world enumeration (DFS over exogenous variables) + forward
#      propagation of structural rules for counterfactual queries (CounterBench):
#      abduction -> action (intervention) -> prediction.
#    * First-order resolution refutation with unification, Skolemisation,
#      CNF conversion, set-of-support and a best-first (UCS by clause weight)
#      given-clause loop (FOLIO).
# =============================================================================
import re
import time
import itertools
from collections import defaultdict

### 1.1 Terms, substitution and unification

In [5]:
# -----------------------------------------------------------------------------
# 1. Terms, substitution and unification
# -----------------------------------------------------------------------------
# A variable is a string starting with '?'.  A constant is any other string.
# A compound term (Skolem function) is a tuple (functor, arg1, ..., argn).


def is_var(t):
    return isinstance(t, str) and t.startswith('?')


def walk(t, s):
    while is_var(t) and t in s:
        t = s[t]
    return t


def occurs(v, t, s):
    t = walk(t, s)
    if t == v:
        return True
    if isinstance(t, tuple):
        return any(occurs(v, a, s) for a in t[1:])
    return False


def unify(a, b, s):
    """Robinson unification. Returns an extended substitution or None."""
    a, b = walk(a, s), walk(b, s)
    if a == b:
        return s
    if is_var(a):
        if occurs(a, b, s):
            return None
        s2 = dict(s); s2[a] = b
        return s2
    if is_var(b):
        if occurs(b, a, s):
            return None
        s2 = dict(s); s2[b] = a
        return s2
    if isinstance(a, tuple) and isinstance(b, tuple) and len(a) == len(b) and a[0] == b[0]:
        for x, y in zip(a[1:], b[1:]):
            s = unify(x, y, s)
            if s is None:
                return None
        return s
    return None


def subst(t, s):
    t = walk(t, s)
    if isinstance(t, tuple):
        return (t[0],) + tuple(subst(a, s) for a in t[1:])
    return t


def unify_args(xs, ys, s):
    if len(xs) != len(ys):
        return None
    for x, y in zip(xs, ys):
        s = unify(x, y, s)
        if s is None:
            return None
    return s

### 1.2 Knowledge base, forward chaining with stratified negation-as-failure, backward chaining

In [6]:
# -----------------------------------------------------------------------------
# 2. Rule base + forward chaining (stratified NAF) + backward chaining
# -----------------------------------------------------------------------------
# Atom    : (pred, (arg1, ..., argn))
# Literal : (positive: bool, Atom)
# Rule    : Rule(body=[Literal], head=[Literal], text=str)

class Rule:
    __slots__ = ('body', 'head', 'text', 'idx')

    def __init__(self, body, head, text='', idx=0):
        self.body, self.head, self.text, self.idx = body, head, text, idx

    def __repr__(self):
        b = ' ∧ '.join(lit_str(l) for l in self.body)
        h = ' ∧ '.join(lit_str(l) for l in self.head)
        return f'{b} → {h}' if self.body else h


def atom_str(a):
    p, args = a
    if not args:                      # propositional atom
        return f'[{p}]'
    return f"{p}({', '.join(term_str(x) for x in args)})"


def term_str(t):
    if isinstance(t, tuple):
        return f"{t[0]}({', '.join(term_str(x) for x in t[1:])})"
    return t


def lit_str(l):
    return ('' if l[0] else '¬') + atom_str(l[1])


class HornKB:
    """Facts + rules with forward chaining.

    mode='CWA': closed world. A negative body literal ¬p(a) succeeds when p(a)
                is not derivable (negation as failure).  Rules are evaluated
                stratum by stratum so NAF is only applied to completed predicates.
    mode='OWA': open world. ¬p(a) must be explicitly derived (classical negation).
    """

    def __init__(self, mode='CWA'):
        self.mode = mode
        self.pos = set()          # ground positive atoms
        self.neg = set()          # ground explicitly negative atoms
        self.rules = []
        self.why = {}             # (sign, atom) -> (rule, [premise literals])
        self.constants = set()
        self.inferences = 0       # number of rule firings (efficiency metric)

    # -- construction ------------------------------------------------------------
    def add_fact(self, lit, text='given'):
        sign, atom = lit
        for a in atom[1]:
            if not is_var(a):
                self.constants.add(a)
        (self.pos if sign else self.neg).add(atom)
        self.why.setdefault((sign, atom), (text, []))

    def add_rule(self, rule):
        rule.idx = len(self.rules)
        self.rules.append(rule)
        for l in rule.body + rule.head:
            for a in l[1][1]:
                if not is_var(a):
                    self.constants.add(a)

    # -- matching ------------------------------------------------------------------
    def _index(self):
        idx = defaultdict(list)
        for a in self.pos:
            idx[(True, a[0])].append(a)
        for a in self.neg:
            idx[(False, a[0])].append(a)
        return idx

    def _match(self, body, s, idx):
        """Generate substitutions satisfying the body (positive literals first)."""
        if not body:
            yield s
            return
        (sign, (pred, args)), rest = body[0], body[1:]
        explicit = sign or self.mode == 'OWA'
        if explicit:
            for fact in idx.get((sign, pred), ()):
                s2 = unify_args(args, fact[1], s)
                if s2 is not None:
                    yield from self._match(rest, s2, idx)
        else:  # CWA negation as failure
            free = sorted({walk(a, s) for a in args if is_var(walk(a, s))})
            doms = [sorted(self.constants)] * len(free)
            for combo in itertools.product(*doms):
                s2 = dict(s)
                for v, c in zip(free, combo):
                    s2[v] = c
                g = (pred, tuple(subst(a, s2) for a in args))
                if g not in self.pos:
                    yield from self._match(rest, s2, idx)

    @staticmethod
    def _order(body):
        return [l for l in body if l[0]] + [l for l in body if not l[0]]

    # -- stratification ----------------------------------------------------------
    def _strata(self):
        preds = set()
        for r in self.rules:
            for l in r.body + r.head:
                preds.add(l[1][0])
        level = {p: 0 for p in preds}
        n = len(preds) + 1
        for _ in range(n + 1):
            changed = False
            for r in self.rules:
                for h in r.head:
                    for b in r.body:
                        need = level[b[1][0]] + (1 if (not b[0] and self.mode == 'CWA') else 0)
                        if level[h[1][0]] < need:
                            level[h[1][0]] = need
                            changed = True
            if not changed:
                break
            if max(level.values(), default=0) > n:   # cycle through negation
                return [list(self.rules)]
        if changed:
            return [list(self.rules)]
        groups = defaultdict(list)
        for r in self.rules:
            groups[max(level[h[1][0]] for h in r.head)].append(r)
        return [groups[k] for k in sorted(groups)]

    # -- forward chaining --------------------------------------------------------
    def saturate(self, max_rounds=200):
        for stratum in self._strata():
            for _ in range(max_rounds):
                idx = self._index()
                new = []
                for r in stratum:
                    for s in self._match(self._order(r.body), {}, idx):
                        self.inferences += 1
                        prem = [(l[0], (l[1][0], tuple(subst(a, s) for a in l[1][1]))) for l in r.body]
                        for h in r.head:
                            g = (h[1][0], tuple(subst(a, s) for a in h[1][1]))
                            if any(is_var(a) for a in g[1]):
                                continue
                            store = self.pos if h[0] else self.neg
                            if g not in store:
                                new.append((h[0], g, r, prem))
                if not new:
                    break
                for sign, g, r, prem in new:
                    store = self.pos if sign else self.neg
                    if g not in store:
                        store.add(g)
                        self.why[(sign, g)] = (r, prem)
        return self

    # -- query -------------------------------------------------------------------
    def holds(self, lit):
        """Returns True / False / None(unknown, only in OWA)."""
        sign, atom = lit
        if self.mode == 'CWA':
            if sign:
                return atom in self.pos
            return atom in self.neg or atom not in self.pos
        if (atom in self.pos) if sign else (atom in self.neg):
            return True
        if (atom in self.neg) if sign else (atom in self.pos):
            return False
        return None

    # -- proof trace (backward reconstruction of the forward derivation) -------
    def explain(self, lit, depth=0, seen=None):
        seen = seen or set()
        sign, atom = lit
        pad = '  ' * depth
        if (sign, atom) in seen:
            return [pad + lit_str(lit)]
        seen = seen | {(sign, atom)}
        w = self.why.get((sign, atom))
        if w is None:
            if not sign and self.mode == 'CWA':
                return [f'{pad}{lit_str(lit)}   [negation as failure: {atom_str(atom)} not derivable]']
            return [f'{pad}{lit_str(lit)}   [not derivable]']
        rule, prem = w
        if isinstance(rule, str):
            return [f'{pad}{lit_str(lit)}   [fact]']
        lines = [f'{pad}{lit_str(lit)}   [rule {rule.idx}: {rule}]']
        for p in prem:
            lines += self.explain(p, depth + 1, seen)
        return lines


def backward_chain(kb, goal, depth=0, max_depth=30, visited=None):
    """Goal-directed proof search (SLD-style with NAF in CWA).  Used as an
    independent check / alternative engine: returns True if goal provable."""
    visited = visited or frozenset()
    sign, atom = goal
    if depth > max_depth or (sign, atom) in visited:
        return False
    visited = visited | {(sign, atom)}
    if kb.mode == 'CWA' and not sign:
        return not backward_chain(kb, (True, atom), depth + 1, max_depth, frozenset())
    if atom in (kb.pos if sign else kb.neg):
        return True
    for r in kb.rules:
        for h in r.head:
            if h[0] != sign or h[1][0] != atom[0]:
                continue
            s = unify_args(h[1][1], atom[1], {})
            if s is None:
                continue
            if _bc_body(kb, kb._order(r.body), s, depth, max_depth, visited):
                return True
    return False


def _bc_body(kb, body, s, depth, max_depth, visited):
    if not body:
        return True
    (sign, (p, args)), rest = body[0], body[1:]
    g_args = tuple(subst(a, s) for a in args)
    free = sorted({a for a in g_args if is_var(a)})
    for combo in itertools.product(*([sorted(kb.constants)] * len(free))):
        s2 = dict(s)
        for v, c in zip(free, combo):
            s2[v] = c
        g = (p, tuple(subst(a, s2) for a in args))
        if backward_chain(kb, (sign, g), depth + 1, max_depth, visited):
            if _bc_body(kb, rest, s2, depth, max_depth, visited):
                return True
    return False

### 1.3 Parsers for rule-based English (first-order rules with variables, and ground rules with fixed connectives)

In [7]:
# -----------------------------------------------------------------------------
# 3. RuleTaker / ProofWriter English -> literals and rules
# -----------------------------------------------------------------------------
GENERIC = {'something', 'someone', 'somebody', 'it', 'they', 'them', 'he', 'she',
           'people', 'things', 'everyone', 'everything', 'anyone', 'anything',
           'person', 'thing', 'you', 'one', 'ones', 'persons', 'animals', 'animal'}
AUX = {'is', 'are', 'was', 'were', 'does', 'do', 'did', 'can', 'cannot', 'has', 'have',
       "isn't", "aren't", "doesn't", "don't", "can't", 'will', 'must', 'would'}
NP_START = {'the', 'a', 'an', 'it', 'them', 'him', 'her', 'something', 'someone',
            'somebody', 'everything', 'everyone', 'they', 'people', 'things'}
VAR = '?x'


def lemma(v):
    v = v.lower()
    irregular = {'has': 'have', 'is': 'be', 'are': 'be', 'does': 'do', 'goes': 'go'}
    if v in irregular:
        return irregular[v]
    if v.endswith('ies') and len(v) > 4:
        return v[:-3] + 'y'
    if v.endswith(('ches', 'shes', 'sses', 'xes', 'zzes')):
        return v[:-2]
    if v.endswith('s') and not v.endswith('ss') and len(v) > 2:
        return v[:-1]
    return v


def singular(w):
    w = w.lower()
    if w.endswith('ies') and len(w) > 4:
        return w[:-3] + 'y'
    if w.endswith('ses') or w.endswith('xes') or w.endswith('ches') or w.endswith('shes'):
        return w[:-2]
    if w.endswith('s') and not w.endswith('ss') and len(w) > 3:
        return w[:-1]
    return w


class ParseError(Exception):
    pass


def norm_np(tokens):
    """Noun phrase -> constant or variable."""
    toks = [t for t in tokens if t]
    if not toks:
        raise ParseError('empty NP')
    low = [t.lower() for t in toks]
    if len(low) == 1 and low[0] in GENERIC:
        return VAR
    if low[0] in ('the', 'a', 'an'):
        low = low[1:]
        if not low:
            raise ParseError('bare article')
        if len(low) == 1 and low[0] in GENERIC:
            return VAR
        if len(low) > 3:
            raise ParseError('NP too long')
        return '_'.join(low)
    if len(toks) == 1 and toks[0][0].isupper():
        return low[0]
    if len(toks) <= 2 and all(t[0].isupper() for t in toks):
        return '_'.join(low)
    raise ParseError('bad NP: ' + ' '.join(toks))


def find_verb(toks):
    low = [t.lower() for t in toks]
    if not toks:
        raise ParseError('empty clause')
    if low[0] in ('the', 'a', 'an'):
        for i in range(2, len(toks)):
            w = low[i]
            if w in AUX:
                return i
            if w.endswith('s') and (i + 1 == len(toks) or low[i + 1] in NP_START or toks[i + 1][0].isupper()):
                return i
        raise ParseError('no verb')
    if len(toks) > 1 and toks[0][0].isupper() and toks[1][0].isupper() and low[0] not in GENERIC:
        return 2
    return 1


def parse_vp(subj, vtoks):
    """Verb phrase applied to subject -> literal."""
    low = [t.lower() for t in vtoks]
    if not low:
        raise ParseError('empty VP')
    pol = True
    v = low[0]
    if v in ('is', 'are', 'was', 'were', "isn't", "aren't"):
        rest = low[1:]
        if v in ("isn't", "aren't"):
            pol = False
        if rest and rest[0] == 'not':
            pol, rest = False, rest[1:]
        if rest and rest[0] in ('a', 'an'):
            rest = rest[1:]
        if not rest:
            raise ParseError('no adjective')
        if len(rest) > 2:
            raise ParseError('adjective phrase too long')
        return (pol, ('_'.join(rest), (subj,)))
    if v in ('can', 'cannot', "can't"):
        rest = low[1:]
        if v != 'can':
            pol = False
        if rest and rest[0] == 'not':
            pol, rest = False, rest[1:]
        if len(rest) != 1:
            raise ParseError('bad can-VP')
        return (pol, ('can_' + lemma(rest[0]), (subj,)))
    if v in ('does', 'do', "doesn't", "don't", 'did'):
        rest = low[1:]
        if v in ("doesn't", "don't"):
            pol = False
        if rest and rest[0] == 'not':
            pol, rest = False, rest[1:]
        if not rest:
            raise ParseError('bad do-VP')
        v = rest[0]
        vtoks = vtoks[len(vtoks) - len(rest):]
        low = rest
    verb = lemma(v)
    obj_toks = vtoks[1:]
    if not obj_toks:
        return (pol, (verb, (subj,)))      # intransitive: "the bell rings"
    return (pol, (verb, (subj, norm_np(obj_toks))))


def parse_clause(text, prev_subj=None):
    """'Bob is big and not red' / 'it chases the cat' -> [literals]."""
    text = text.strip().strip(',').strip()
    parts = [p.strip() for p in re.split(r',?\s+and\s+|,\s+', text) if p.strip()]
    lits = []
    subj = prev_subj
    for p in parts:
        toks = p.split()
        low0 = toks[0].lower()
        has_subject = (low0 in GENERIC or low0 in ('the', 'a', 'an') or
                       (toks[0][0].isupper() and low0 not in AUX and low0 != 'not'))
        if has_subject and len(toks) >= 2:
            i = find_verb(toks)
            subj = norm_np(toks[:i])
            lits.append(parse_vp(subj, toks[i:]))
        else:
            if subj is None:
                raise ParseError('missing subject')
            if low0 in AUX or (low0.endswith('s') and len(toks) > 1):
                lits.append(parse_vp(subj, toks))
            else:   # bare adjective continuing a copula: "big and not round"
                lits.append(parse_vp(subj, ['is'] + toks))
    return lits, subj


def _adj_list(text):
    """'red, cold' / 'big and kind' -> ['red','cold']"""
    words = [w for w in re.split(r',\s*|\s+and\s+|\s+', text.strip()) if w]
    return [w.lower() for w in words if w.lower() not in ('all', 'the')]


def parse_rule_sentence(s):
    """Returns a Rule, a list of fact literals, or raises ParseError."""
    s = s.strip().rstrip('.').strip()
    low = s.lower()
    # If ... then ...
    m = re.match(r'^if\s+(.+?)\s*,?\s+then\s+(.+)$', s, re.I) or \
        re.match(r'^if\s+(.+?),\s+(.+)$', s, re.I)
    if m:
        body, subj = parse_clause(m.group(1))
        head, _ = parse_clause(m.group(2), subj)
        return Rule(body, head, s)
    # All/Every red, cold things are round
    m = re.match(r'^(?:all\s+|every\s+)?(.+?)\s+(things|people|persons|animals|ones)\s+(are|is|do|does|can|cannot)\b\s*(.*)$', s, re.I)
    if m and not m.group(1).lower() in ('the',):
        adjs = _adj_list(m.group(1))
        if adjs and all(re.fullmatch(r"[a-z\-]+", a) for a in adjs) and len(adjs) <= 4:
            body = []
            for a in adjs:
                body.append((True, (a, (VAR,))))
            head, _ = parse_clause('they ' + m.group(3) + ' ' + m.group(4))
            return Rule(body, head, s)
    # All cats are animals / All birds can fly
    m = re.match(r'^all\s+(\w+)\s+(are|can|cannot|do|does)\b\s*(.*)$', s, re.I)
    if m:
        noun = singular(m.group(1))
        rest = m.group(3).strip()
        if m.group(2).lower() == 'are' and re.fullmatch(r'(?:not\s+)?[a-z]+s', rest, re.I) and not rest.endswith('ss'):
            rest = re.sub(r'(\w+)$', lambda w: singular(w.group(1)), rest)   # "are mammals" -> mammal
        head, _ = parse_clause('they ' + m.group(2) + ' ' + rest)
        return Rule([(True, (noun, (VAR,)))], head, s)
    # Things that are X are Y / People who are X are Y / Someone who is X is Y
    m = re.match(r'^(?:all\s+)?(?:things|people|someone|something|anyone|anything|everyone|everything)\s+(?:that|who|which)\s+(.+?)\s+(are|is)\s+(?:also\s+)?(.+)$', s, re.I)
    if m:
        body, _ = parse_clause('they ' + m.group(1))
        head, _ = parse_clause('they is ' + m.group(3))
        return Rule(body, head, s)
    lits, _ = parse_clause(s)
    return lits


def split_sentences(text):
    text = text.replace('\n', ' ')
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if p.strip()]


QUESTION_PREFIX = re.compile(r'^.*?(true, false, or unknown\?|true or false\?|is it true that)\s*', re.I)


def parse_ruletaker(context, statement, mode='CWA'):
    kb = HornKB(mode)
    unparsed = []
    for sent in split_sentences(context):
        try:
            r = parse_rule_sentence(sent)
        except ParseError:
            r = None
        if r is None:
            unparsed.append(sent)
            continue
        if isinstance(r, Rule):
            if not r.body or not r.head:
                unparsed.append(sent)
                continue
            kb.add_rule(r)
        else:
            for lit in r:
                if any(is_var(a) for a in lit[1][1]):
                    unparsed.append(sent)
                    break
                kb.add_fact(lit, sent)
    stmt = QUESTION_PREFIX.sub('', statement.strip()).rstrip('.?').strip()
    try:
        q, _ = parse_clause(stmt)
    except ParseError:
        q = None
    return kb, q, unparsed



# --- 3b. Ground (propositional) rules written with fixed connective phrases ---------
# e.g. "Assuming the river is fast, we can say the house volunteers."
#      "Given that A, it follows that B."  "Based on A, it is evident that B."
# Each simple clause becomes one propositional atom (a normalised string); the
# same forward chainer is then used with closed-world semantics.
RULE_TEMPLATES = [
    re.compile(r'^if\s+(.+?),?\s+then\s+(.+)$', re.I),
    re.compile(r'^assuming\s+(?:that\s+)?(.+?),\s*we\s+can\s+(?:say|infer|deduce)\s+(?:that\s+)?(.+)$', re.I),
    re.compile(r'^based\s+on\s+(?:the\s+fact\s+that\s+)?(.+?),\s*it\s+is\s+(?:evident|clear)\s+that\s+(.+)$', re.I),
    re.compile(r'^given\s+(?:that\s+)?(.+?),\s*it\s+follows\s+that\s+(.+)$', re.I),
    re.compile(r'^when(?:ever)?\s+(.+?),\s*we\s+can\s+conclude\s+(?:that\s+)?(.+)$', re.I),
]
LEAD_IN = re.compile(r'^(?:if|assuming|based on|given that|given|when|whenever|since|because|provided that|as long as|once)\s+(.+?),\s*(.+)$', re.I)
CONSEQ_LEAD = re.compile(r'^(?:then|so|thus|hence|therefore|we can say|we can conclude|we can infer|it is evident that|'
                         r'it is clear that|it follows that|this means that|it means that|that)\s+', re.I)
PROP_SKIP = {'the', 'a', 'an', 'does', 'do', 'did'}
NEEDS_VARIABLES = re.compile(r'\b(someone|somebody|something|anyone|anything|everyone|everything|people|things|'
                             r'they|them|it|all|every|each|some|no|any|either|those|who|which)\b', re.I)


def prop_atom(text):
    """Clause -> signed propositional literal, e.g. 'the house does not volunteer'
    -> (False, ('house volunteer', ()))."""
    words = re.sub(r"[^\w\s']", ' ', text.lower()).split()
    pol, kept = True, []
    for w in words:
        if w == 'not':
            pol = not pol
            continue
        if w.endswith("n't"):
            pol = not pol
            w = {"won't": 'will', "can't": 'can'}.get(w, w[:-3])
        if w in PROP_SKIP:
            continue
        kept.append(lemma(w))
    if not kept:
        raise ParseError('empty clause')
    return (pol, (' '.join(kept), ()))


def prop_clauses(text):
    """'A and B' -> [A, B] when both sides look like full clauses."""
    parts = [p.strip() for p in re.split(r'\s+and\s+', text.strip()) if p.strip()]
    if len(parts) > 1 and all(len(p.split()) >= 2 for p in parts):
        return [prop_atom(p) for p in parts]
    return [prop_atom(text)]


def parse_prop_rule(sent):
    s = sent.strip().rstrip('.').strip()
    for pat in RULE_TEMPLATES:
        m = pat.match(s)
        if m:
            return m.group(1), m.group(2)
    m = LEAD_IN.match(s)
    if m:
        concl = m.group(2)
        while CONSEQ_LEAD.match(concl):
            concl = CONSEQ_LEAD.sub('', concl, count=1)
        return m.group(1), concl
    return None


def parse_propositional(context, statement, mode='CWA'):
    kb = HornKB(mode)
    unparsed, n_rules = [], 0
    text = re.sub(r'\b(facts|rules|context|theory)\s*:', ' ', context, flags=re.I)
    for sent in split_sentences(text):
        try:
            rule = parse_prop_rule(sent)
            if rule:
                cond, concl = rule
                if NEEDS_VARIABLES.search(cond) or NEEDS_VARIABLES.search(concl):
                    raise ParseError('rule needs variables')
                for alt in re.split(r'\s+or\s+', cond):            # "if A or B then C" -> two rules
                    kb.add_rule(Rule(prop_clauses(alt), prop_clauses(concl), sent.strip()))
                n_rules += 1
            else:
                if len(sent.split()) > 9 or NEEDS_VARIABLES.search(sent) or re.match(r'^(if|when|unless)\b', sent, re.I):
                    raise ParseError('not a simple fact')
                for lit in prop_clauses(sent.rstrip('.')):
                    kb.add_fact(lit, sent.strip())
        except ParseError:
            unparsed.append(sent)
    stmt = QUESTION_PREFIX.sub('', statement.strip()).rstrip('.?').strip()
    try:
        q = prop_clauses(stmt) if stmt else None
    except ParseError:
        q = None
    if n_rules == 0:
        q = None
    return kb, q, unparsed

### 1.4 Counterfactual reasoning (CounterBench): structural rules + possible-world search

In [8]:
# -----------------------------------------------------------------------------
# 4. CounterBench: causal clauses -> structural rules -> counterfactual search
# -----------------------------------------------------------------------------
class CausalModel:
    """Deterministic structural causal model over Boolean variables.

    equations[v] = list of (op, [(var, polarity)], out_polarity) disjuncts;
        v is true iff any disjunct fires.  op in {'and', 'or', 'id'}.
    priors[root] = P(root = 1); default 0.5.
    """

    def __init__(self):
        self.equations = defaultdict(list)
        self.priors = {}
        self.vars = []
        self.evidence = {}          # observed facts in the factual world
        self.trace = []

    def var(self, name):
        name = re.sub(r'\s+', ' ', name.strip().lower())
        name = re.sub(r'\s+(occurs?|occurred|happens?)$', '', name)
        if name not in self.vars:
            self.vars.append(name)
        return name

    def roots(self):
        return [v for v in self.vars if v not in self.equations]

    def eq_str(self, v):
        out = []
        for op, ins, outpol in self.equations[v]:
            lits = [('' if p else '¬') + a for a, p in ins]
            e = (' ∧ ' if op == 'and' else ' ∨ ').join(lits)
            e = e if len(lits) == 1 else f'({e})'
            out.append(e if outpol else f'¬{e}')
        return f"{v} := {' ∨ '.join(out)}"

    def evaluate(self, root_vals, interventions):
        val = dict(root_vals)
        val.update(interventions)

        def get(v, stack=()):
            if v in val:
                return val[v]
            if v in stack:                       # cycle guard
                return False
            res = False
            for op, ins, outpol in self.equations[v]:
                xs = [get(a, stack + (v,)) == p for a, p in ins]
                fired = all(xs) if op in ('and', 'id') else any(xs)
                # "A causes not B": B = not A
                res = res or (fired if outpol else not fired)
            val[v] = res
            return res

        for v in self.vars:
            get(v)
        return val


def _cb_literal(txt, cm):
    txt = txt.strip()
    m = re.match(r'^not\s+(.+)$', txt, re.I)
    if m:
        return cm.var(m.group(1)), False
    m = re.match(r'^(.+?)\s+(?:does\s+)?not\s+occurs?$', txt, re.I)
    if m:
        return cm.var(m.group(1)), False
    return cm.var(txt), True


def _cb_split_clauses(body):
    pieces = []
    for chunk in re.split(r',\s*', body):
        chunk = re.sub(r'^and\s+', '', chunk.strip(), flags=re.I)
        if not chunk:
            continue
        # two clauses joined by " and " without comma: "A causes B and B causes C"
        verbs = list(re.finditer(r'\b(causes|cause|has a direct effect on|have a direct effect on)\b', chunk))
        if len(verbs) >= 2:
            cut = None
            for m in re.finditer(r'\s+and\s+', chunk):
                if verbs[0].end() <= m.start() < verbs[1].start():
                    cut = m
            if cut:
                pieces += [chunk[:cut.start()], chunk[cut.end():]]
                continue
        pieces.append(chunk)
    return pieces


def parse_counterbench(context, question):
    cm = CausalModel()
    text = (context + ' ' + question).replace('\n', ' ')
    sentences = [s.strip() for s in re.split(r'(?<!\d)[.?](?!\d)', text) if s.strip()]
    query = None
    for sent in sentences:
        low = sent.lower()
        for m in re.finditer(r'([A-Za-z][\w ]*?)\s*~\s*bern\(\s*([\d.]+)\s*\)', sent, re.I):
            cm.priors[cm.var(m.group(1).split()[-1])] = float(m.group(2))
        if low.startswith('imagine'):
            continue
        if 'we know that' in low:
            body = sent[low.index('we know that') + len('we know that'):]
            for cl in _cb_split_clauses(body):
                if re.search(r'bern\(', cl, re.I):
                    continue
                m = re.match(r'^(.*?)\s+(?:together\s+)?(causes|cause|has a direct effect on|have a direct effect on)\s+(not\s+)?(.+)$', cl.strip(), re.I)
                if not m:
                    continue
                lhs = re.sub(r'\s+together$', '', m.group(1).strip(), flags=re.I)
                out = cm.var(m.group(4))
                outpol = m.group(3) is None
                if re.search(r'\s+or\s+', lhs):
                    op, parts = 'or', re.split(r'\s+or\s+', lhs)
                elif re.search(r'\s+and\s+', lhs):
                    op, parts = 'and', re.split(r'\s+and\s+', lhs)
                else:
                    op, parts = 'id', [lhs]
                ins = [_cb_literal(p, cm) for p in parts]
                cm.equations[out].append((op, ins, outpol))
            continue
        m = re.match(r'^we observed\s+(.+)$', sent.strip(), re.I)
        if m:
            for part in re.split(r'\s+and\s+|,\s*', m.group(1)):
                if part.strip():
                    v, p = _cb_literal(re.sub(r'\s+(occurs?|occurred)$', '', part.strip(), flags=re.I)
                                       if not re.search(r'not\s+occurs?$', part.strip(), re.I) else part, cm)
                    cm.evidence[v] = p
            continue
        m = re.search(r'would\s+(.+?)\s+(not\s+)?occur\s+if\s+(.+?)\s+instead\s+of\s+(.+)$', sent, re.I)
        if m:
            query = (cm.var(m.group(1)), m.group(2) is None,
                     [_cb_literal(x, cm) for x in re.split(r'\s+and\s+|,\s*', m.group(3)) if x.strip()],
                     [_cb_literal(x, cm) for x in re.split(r'\s+and\s+|,\s*', m.group(4)) if x.strip()])
            continue
        m = re.search(r'assume\s+(.+?),?\s+and\s+based\s+on\s+this\s+assumption,?\s+further\s+suppose\s+(.+)$', sent, re.I)
        if m:
            ivs = [_cb_literal(x, cm) for x in re.split(r'\s+and\s+', m.group(1))] + \
                  [_cb_literal(x, cm) for x in re.split(r'\s+and\s+', m.group(2))]
            cm._nested = ivs
            continue
        m = re.search(r'would\s+(.+?)\s+(not\s+)?occur$', sent, re.I)
        if m and getattr(cm, '_nested', None):
            ivs = cm._nested
            query = (cm.var(m.group(1)), m.group(2) is None, ivs, [(v, not p) for v, p in ivs])
    return cm, query


def counterfactual(cm, query):
    """Abduction -> action -> prediction, by exhaustive DFS over exogenous
    (root) assignments weighted by their priors.  Returns (answer, P(yes))."""
    target, tpol, interventions, factual = query
    factual_ev = dict(cm.evidence)
    for v, p in factual:
        factual_ev.setdefault(v, p)
    roots = cm.roots()
    num = den = 0.0
    worlds = 0
    for relax in (0, 1, 2):
        ev = factual_ev if relax == 0 else (cm.evidence if relax == 1 else {})
        num = den = 0.0
        for combo in itertools.product((False, True), repeat=len(roots)):
            rv = dict(zip(roots, combo))
            fw = cm.evaluate(rv, {})
            if any(fw.get(v) != p for v, p in ev.items()):
                continue                                   # inconsistent with observations
            w = 1.0
            for r, b in rv.items():
                pr = cm.priors.get(r, 0.5)
                w *= pr if b else (1 - pr)
            cw = cm.evaluate(rv, {v: p for v, p in interventions})
            worlds += 1
            den += w
            if cw.get(target) == tpol:
                num += w
        if den > 0:
            break
    p_yes = num / den if den else 0.0
    cm.trace = [f'roots={roots}', f'evidence={factual_ev}', f'do={dict(interventions)}',
                f'consistent worlds={worlds}', f'P({"" if tpol else "¬"}{target} | evidence, do)={p_yes:.3f}']
    return p_yes > 0.5, p_yes


def looks_counterbench(context, statement):
    t = (context + ' ' + statement).lower()
    return bool(re.search(r'\bcauses?\b', t) and re.search(r'would .+ occur', t)) or \
        'we know that' in t and 'cause' in t

### 1.5 First-order logic: parser, CNF / Skolemisation, resolution refutation prover

In [9]:
# -----------------------------------------------------------------------------
# 5. First-order logic: parsing, CNF conversion, resolution refutation
# -----------------------------------------------------------------------------
FOL_TOKEN = re.compile(r"\s*(∀|∃|¬|∧|∨|→|↔|⊕|≠|<->|->|=>|<=>|&|\||~|!|=|\(|\)|,|[\w\-'’]+)")
OPS = {'¬': 'not', '~': 'not', '!': 'not', '∧': 'and', '&': 'and', '∨': 'or', '|': 'or',
       '→': 'imp', '->': 'imp', '=>': 'imp', '↔': 'iff', '<->': 'iff', '<=>': 'iff', '⊕': 'xor'}
QUANTS = {'∀': 'all', '∃': 'ex', 'forall': 'all', 'exists': 'ex'}


class FOLParseError(Exception):
    pass


def fol_tokenize(s):
    s = s.replace('⟶', '→').replace('⇒', '→').replace('⟷', '↔').replace('⇔', '↔')
    toks, pos = [], 0
    s = s.strip()
    while pos < len(s):
        m = FOL_TOKEN.match(s, pos)
        if not m or m.end() == pos:
            if s[pos].isspace():
                pos += 1
                continue
            pos += 1          # skip unknown symbol
            continue
        toks.append(m.group(1))
        pos = m.end()
    return toks


class FOLParser:
    def __init__(self, text):
        self.t = fol_tokenize(text)
        self.i = 0

    def peek(self, k=0):
        return self.t[self.i + k] if self.i + k < len(self.t) else None

    def take(self):
        tok = self.peek()
        self.i += 1
        return tok

    def parse(self):
        f = self.p_iff()
        # annotation slips: stray ')' are skipped, and parsing resumes with the
        # operator that follows (or a conjunction if two formulas are juxtaposed)
        binop = {'∧': 'and', '&': 'and', '∨': 'or', '|': 'or', '⊕': 'xor', '→': 'imp', '->': 'imp',
                 '=>': 'imp', '↔': 'iff', '<->': 'iff', '<=>': 'iff'}
        while self.peek() is not None:
            tok = self.peek()
            if tok == ')':
                self.i += 1
                continue
            if tok in binop:
                self.i += 1
                if self.peek() is None:
                    break
                f = (binop[tok], f, self.p_iff())
                continue
            f = ('and', f, self.p_iff())
        return f

    def p_iff(self):
        f = self.p_imp()
        while self.peek() in ('↔', '<->', '<=>'):
            self.take()
            f = ('iff', f, self.p_imp())
        return f

    def p_imp(self):
        f = self.p_xor()
        if self.peek() in ('→', '->', '=>'):
            self.take()
            return ('imp', f, self.p_imp())
        return f

    def p_xor(self):
        f = self.p_or()
        while self.peek() == '⊕':
            self.take()
            f = ('xor', f, self.p_or())
        return f

    def p_or(self):
        f = self.p_and()
        while self.peek() in ('∨', '|'):
            self.take()
            f = ('or', f, self.p_and())
        return f

    def p_and(self):
        f = self.p_unary()
        while self.peek() in ('∧', '&'):
            self.take()
            f = ('and', f, self.p_unary())
        return f

    def p_unary(self):
        tok = self.peek()
        if tok is None:
            raise FOLParseError('unexpected end')
        if tok in ('¬', '~', '!'):
            self.take()
            return ('not', self.p_unary())
        if tok in QUANTS:
            q = QUANTS[self.take()]
            vs = []
            if self.peek() is None or not re.fullmatch(r'\w+', self.peek()):
                raise FOLParseError('quantifier without variable')
            vs.append(self.take())
            # "∀x, y (...)" or "∀x y (...)" : extra short variable names
            while True:
                if self.peek() == ',' and self.peek(1) and re.fullmatch(r'[a-z]\d?', self.peek(1)):
                    self.take(); vs.append(self.take())
                else:
                    break
            body = self.p_iff() if self.peek() not in (None, ')') else ('true',)
            for v in reversed(vs):
                body = (q, v, body)
            return body
        if tok == '(':
            self.take()
            f = self.p_iff()
            if self.peek() == ')':
                self.take()
            return f
        return self.p_atom()

    def p_term(self):
        tok = self.take()
        if tok is None or tok in OPS or tok in ('(', ')', ','):
            raise FOLParseError(f'bad term {tok}')
        if self.peek() == '(':
            self.take()
            args = []
            while self.peek() not in (')', None) and self.peek() not in OPS and self.peek() not in QUANTS:
                args.append(self.p_term())
                if self.peek() == ',':
                    self.take()
            if self.peek() == ')':
                self.take()
            return ('fn', tok, tuple(args))
        return tok

    def p_atom(self):
        name = self.take()
        if name in OPS or name in (')', ','):
            raise FOLParseError(f'unexpected {name}')
        if self.peek() == '(':
            self.take()
            args = []
            while self.peek() not in (')', None) and self.peek() not in OPS and self.peek() not in QUANTS:
                if self.peek() == '(':           # stray '(' inside an argument list
                    self.take()
                    continue
                args.append(self.p_term())
                if self.peek() == ',':
                    self.take()
            if self.peek() == ')':
                self.take()
            left = ('fn', name, tuple(args))
        else:
            left = name
        if self.peek() in ('=', '≠'):
            op = self.take()
            right = self.p_term()
            a = ('atom', '=', (left, right))
            return a if op == '=' else ('not', a)
        if isinstance(left, tuple):
            return ('atom', name, left[2])
        return ('atom', name, ())


# --- variable binding, NNF, Skolemisation, CNF --------------------------------
_fresh = itertools.count()


def bind_vars(f, env=None, free=None):
    """Replace quantified variable names by unique '?v' variables and turn
    unbound single-letter arguments (annotation slips) into free variables."""
    env = env or {}
    if free is None:
        free = {}
    op = f[0]
    if op in ('all', 'ex'):
        nv = f'?{f[1]}{next(_fresh)}'
        return (op, nv, bind_vars(f[2], {**env, f[1]: nv}, free))
    if op == 'atom':
        return ('atom', f[1], tuple(_bind_term(a, env, free) for a in f[2]))
    if op == 'true':
        return f
    if op == 'not':
        return ('not', bind_vars(f[1], env, free))
    return (op, bind_vars(f[1], env, free), bind_vars(f[2], env, free))


def _bind_term(t, env, free):
    if isinstance(t, tuple):
        return (t[1],) + tuple(_bind_term(a, env, free) for a in t[2])
    if t in env:
        return env[t]
    if re.fullmatch(r'[a-z]\d?', t):            # unbound variable-looking symbol
        if t not in free:
            free[t] = f'?{t}{next(_fresh)}'
        return free[t]
    return t.lower()


def nnf(f, neg=False):
    op = f[0]
    if op == 'atom':
        return ('not', f) if neg else f
    if op == 'true':
        return ('false',) if neg else f
    if op == 'false':
        return ('true',) if neg else f
    if op == 'not':
        return nnf(f[1], not neg)
    if op == 'and':
        return ('or' if neg else 'and', nnf(f[1], neg), nnf(f[2], neg))
    if op == 'or':
        return ('and' if neg else 'or', nnf(f[1], neg), nnf(f[2], neg))
    if op == 'imp':
        return nnf(('or', ('not', f[1]), f[2]), neg)
    if op == 'iff':
        a, b = f[1], f[2]
        return nnf(('and', ('imp', a, b), ('imp', b, a)), neg)
    if op == 'xor':
        a, b = f[1], f[2]
        return nnf(('and', ('or', a, b), ('or', ('not', a), ('not', b))), neg)
    if op == 'all':
        return ('ex' if neg else 'all', f[1], nnf(f[2], neg))
    if op == 'ex':
        return ('all' if neg else 'ex', f[1], nnf(f[2], neg))
    raise ValueError(op)


def skolemize(f, univ=(), s=None):
    s = s or {}
    op = f[0]
    if op == 'all':
        return skolemize(f[2], univ + (f[1],), s)
    if op == 'ex':
        sk = f'sk{next(_fresh)}'
        s = {**s, f[1]: (sk,) + univ if univ else sk}
        return skolemize(f[2], univ, s)
    if op == 'atom':
        return ('atom', f[1], tuple(subst(a, s) for a in f[2]))
    if op == 'not':
        return ('not', skolemize(f[1], univ, s))
    if op in ('true', 'false'):
        return f
    return (op, skolemize(f[1], univ, s), skolemize(f[2], univ, s))


MAX_CNF = 256


def cnf_clauses(f):
    """Formula in NNF without quantifiers -> list of clauses (frozensets)."""
    op = f[0]
    if op == 'atom':
        return [frozenset([(True, f[1], f[2])])]
    if op == 'not':
        a = f[1]
        return [frozenset([(False, a[1], a[2])])]
    if op == 'true':
        return []
    if op == 'false':
        return [frozenset()]
    if op == 'and':
        return cnf_clauses(f[1]) + cnf_clauses(f[2])
    if op == 'or':
        A, B = cnf_clauses(f[1]), cnf_clauses(f[2])
        if len(A) * len(B) > MAX_CNF:
            raise FOLParseError('CNF blow-up')
        return [a | b for a in A for b in B]
    raise ValueError(op)


def fol_to_clauses(text_or_formula, negate=False):
    f = FOLParser(text_or_formula).parse() if isinstance(text_or_formula, str) else text_or_formula
    free = {}
    f = bind_vars(f, {}, free)
    if negate:
        # free variables of a goal are universal: ¬∀x φ = ∃x ¬φ -> skolem constants
        for v in free.values():
            f = ('all', v, f)
        f = ('not', f)
    f = skolemize(nnf(f))
    out = []
    for c in cnf_clauses(f):
        if any((not s, p, a) in c for (s, p, a) in c):
            continue                          # tautology
        out.append(c)
    return out


# --- resolution prover ------------------------------------------------------------
def term_depth(t):
    if isinstance(t, tuple):
        return 1 + max((term_depth(a) for a in t[1:]), default=0)
    return 0


def clause_weight(c):
    return sum(1 + sum(_tsize(a) for a in l[2]) for l in c)


def _tsize(t):
    return 1 + sum(_tsize(a) for a in t[1:]) if isinstance(t, tuple) else 1


def rename(clause, tag):
    m = {}

    def rn(t):
        if is_var(t):
            if t not in m:
                m[t] = f'{t}_{tag}'
            return m[t]
        if isinstance(t, tuple):
            return (t[0],) + tuple(rn(a) for a in t[1:])
        return t
    return frozenset((s, p, tuple(rn(a) for a in args)) for (s, p, args) in clause)


def canonical(clause):
    """Variable-renamed canonical form used to detect duplicate clauses."""
    lits = sorted(clause, key=lambda l: (l[1], l[0], str([a if not is_var(a) else '?' for a in l[2]])))
    m = {}

    def rn(t):
        if is_var(t):
            m.setdefault(t, f'?{len(m)}')
            return m[t]
        if isinstance(t, tuple):
            return (t[0],) + tuple(rn(a) for a in t[1:])
        return t
    return frozenset((s, p, tuple(rn(a) for a in args)) for (s, p, args) in lits)


def subsumes(c, d):
    """Does clause c subsume d (∃θ: cθ ⊆ d)?  Small backtracking matcher."""
    if len(c) > len(d):
        return False
    c, d = list(c), list(d)

    def match(i, s):
        if i == len(c):
            return True
        sc, pc, ac = c[i]
        for sd, pd, ad in d:
            if sc == sd and pc == pd and len(ac) == len(ad):
                s2 = _match_args(ac, ad, s)
                if s2 is not None and match(i + 1, s2):
                    return True
        return False
    return match(0, {})


def _match_args(xs, ys, s):
    for x, y in zip(xs, ys):
        s = _match_term(x, y, s)
        if s is None:
            return None
    return s


def _match_term(x, y, s):
    if is_var(x):
        if x in s:
            return s if s[x] == y else None
        s2 = dict(s); s2[x] = y
        return s2
    if isinstance(x, tuple):
        if isinstance(y, tuple) and x[0] == y[0] and len(x) == len(y):
            return _match_args(x[1:], y[1:], s)
        return None
    return s if x == y else None


class ResolutionProver:
    """Set-of-support resolution refutation with a best-first (lightest clause
    first, i.e. uniform-cost) given-clause loop, binary resolution + factoring,
    tautology deletion and forward subsumption."""

    def __init__(self, max_given=250, max_clauses=6000, time_limit=0.4, max_depth=3, max_len=10):
        self.max_given, self.max_clauses = max_given, max_clauses
        self.time_limit, self.max_depth, self.max_len = time_limit, max_depth, max_len
        self.steps = 0
        self.proof = None

    def prove(self, kb_clauses, goal_clauses):
        t0 = time.perf_counter()
        parent = {}
        usable = []                      # processed clauses
        seen = set()
        sos = []                         # (weight, id, clause)
        counter = itertools.count()

        def add(c, src, into):
            k = canonical(c)
            if k in seen:
                return False
            seen.add(k)
            parent[c] = src
            into(c)
            return True

        for c in kb_clauses:
            add(c, ('kb',), usable.append)
        for c in goal_clauses:
            add(c, ('goal',), lambda c: sos.append((clause_weight(c), next(counter), c)))
            if not c:
                self.proof = c
                return True, parent
        idx = defaultdict(list)          # (sign, pred) -> clauses in usable
        for c in usable:
            for l in c:
                idx[(l[0], l[1])].append(c)
        given = 0
        while sos and given < self.max_given and len(seen) < self.max_clauses:
            if time.perf_counter() - t0 > self.time_limit:
                break
            sos.sort(key=lambda x: (x[0], x[1]))
            _, _, g = sos.pop(0)
            given += 1
            new = []
            # factoring
            for l1, l2 in itertools.combinations(list(g), 2):
                if l1[0] == l2[0] and l1[1] == l2[1]:
                    s = unify_args(l1[2], l2[2], {})
                    if s is not None:
                        f = frozenset((x[0], x[1], tuple(subst(a, s) for a in x[2])) for x in g if x != l2)
                        new.append((f, ('factor', g)))
            # binary resolution with all processed clauses
            for lit in g:
                for c in list(idx.get((not lit[0], lit[1]), ())):
                    cr = rename(c, self.steps)
                    self.steps += 1
                    for l2 in cr:
                        if l2[0] == lit[0] or l2[1] != lit[1]:
                            continue
                        s = unify_args(lit[2], l2[2], {})
                        if s is None:
                            continue
                        r = frozenset((x[0], x[1], tuple(subst(a, s) for a in x[2]))
                                      for x in (g - {lit}) | (cr - {l2}))
                        new.append((r, ('res', g, c)))
            usable.append(g)
            for l in g:
                idx[(l[0], l[1])].append(g)
            for r, src in new:
                if not r:
                    parent[r] = src
                    self.proof = r
                    return True, parent
                if len(r) > self.max_len:
                    continue
                if any((not s, p, a) in r for (s, p, a) in r):
                    continue
                if any(term_depth(a) > self.max_depth for l in r for a in l[2]):
                    continue
                if any(subsumes(u, r) for u in usable if len(u) <= len(r) and len(u) <= 3):
                    continue
                add(r, src, lambda c: sos.append((clause_weight(c), next(counter), c)))
        return False, parent


def fol_clause_str(c):
    if not c:
        return '⊥ (empty clause: contradiction)'
    return ' ∨ '.join(('' if s else '¬') + f"{p}({', '.join(term_str(a) for a in args)})" for s, p, args in c)


def proof_trace(parent, max_lines=25):
    lines = []

    def rec(c, depth):
        if len(lines) >= max_lines:
            return
        src = parent.get(c, ('kb',))
        tag = src[0]
        lines.append('  ' * depth + f'{fol_clause_str(c)}   [{tag}]')
        for p in src[1:]:
            rec(p, depth + 1)
    rec(frozenset(), 0)
    return lines


def fol_entails(premises_fol, conclusion_fol, prover=None):
    """KB ⊨ φ  iff  KB ∪ {¬φ} is unsatisfiable (resolution refutation)."""
    prover = prover or ResolutionProver()
    kb = []
    for p in premises_fol:
        try:
            kb += fol_to_clauses(p)
        except (FOLParseError, ValueError, RecursionError):
            continue
    goal = fol_to_clauses(conclusion_fol, negate=True)
    eq = any(l[1] == '=' for c in kb + goal for l in c)
    if eq:   # minimal equality axioms: reflexivity and symmetry
        kb.append(frozenset([(True, '=', ('?e1', '?e1'))]))
        kb.append(frozenset([(False, '=', ('?e2', '?e3')), (True, '=', ('?e3', '?e2'))]))
    ok, parent = prover.prove(kb, goal)
    return ok, parent, prover

### 1.6 Fallback parsers for freer English (paraphrased rules, FOLIO-style sentences)

In [10]:
# -----------------------------------------------------------------------------
# 6. Fallback parsers for freer English
#    6a. Lexicon-based extraction for paraphrased RuleTaker text (ParaRules /
#        NatLang configs): attributes, entities, negation, condition/conclusion.
#    6b. Shallow NL -> FOL for FOLIO-style sentences when no FOL is supplied.
# -----------------------------------------------------------------------------
NEG_WORDS = {'not', "n't", 'never', 'no', 'neither', 'nor', 'cannot'}
GENERIC_MARKERS = {'someone', 'somebody', 'people', 'person', 'persons', 'anyone', 'anybody',
                   'things', 'something', 'anything', 'everyone', 'everybody', 'those', 'whoever',
                   'guy', 'guys', 'individual', 'individuals', 'one', 'ones', 'folks', 'you'}
CONCL_MARKERS = re.compile(r"\b(then|they are|they're|they will be|will be|would be|are also|is also|is|are|be|"
                           r"makes? (?:you|them|someone|people|a person)|seems?|tends? to be|must be|turns?|becomes?|get|gets)\b", re.I)


def natlang_fallback(sentences, kb, lexicon, entities):
    added = 0
    for sent in sentences:
        toks = re.findall(r"[A-Za-z']+", sent)
        low = [t.lower() for t in toks]
        attrs = []
        for i, w in enumerate(low):
            if w in lexicon:
                neg = any(x in NEG_WORDS or x.endswith("n't") for x in low[max(0, i - 3):i])
                attrs.append((i, w, not neg))
        if not attrs:
            continue
        names = [(i, low[i]) for i, t in enumerate(toks) if low[i] in entities or
                 (t[0].isupper() and i > 0 and low[i] not in GENERIC_MARKERS)]
        generic = any(w in GENERIC_MARKERS for w in low) or low[0] in ('if', 'when', 'whenever', 'a', 'an', 'all', 'any', 'every') and not names
        if generic and not names:
            text = ' '.join(low)
            cut = None
            for m in CONCL_MARKERS.finditer(text):
                cut = m
            if cut is None:
                continue
            pos = len(text[:cut.start()].split())
            body = [(True if p else False, (a, (VAR,))) for i, a, p in attrs if i < pos]
            head = [(True if p else False, (a, (VAR,))) for i, a, p in attrs if i >= pos]
            if body and head:
                kb.add_rule(Rule(body, head, sent))
                added += 1
        elif names:
            for i, a, p in attrs:
                owner = [n for j, n in names if j < i]
                ent = owner[-1] if owner else names[0][1]
                kb.add_fact((p, (a, (ent,))), sent)
                added += 1
    return added


FOL_STOP = {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'who', 'that', 'which',
            'also', 'does', 'do', 'did', 'has', 'have', 'had', 'their', 'its', 'his', 'her', 'they', 'them',
            'it', 'he', 'she', 'to', 'of', 'all', 'every', 'each', 'any', 'some', 'people', 'person',
            'someone', 'somebody', 'everyone', 'everybody', 'things', 'thing', 'and', 'either', 'or', 'if',
            'then', 'not', 'no', 'never', 'will', 'can', 'would', 'should', 'must', 'there', 'those'}
PLURAL_SUBJ = {'people', 'persons', 'humans', 'someone', 'everyone', 'everybody', 'anyone', 'things',
               'something', 'everything', 'person', 'individuals'}


def phrase_key(words):
    keep = [lemma(w) if w.endswith('s') and len(w) > 3 else w for w in words if w not in FOL_STOP]
    return '_'.join(keep)


def nl_to_fol(sentence):
    """Very shallow semantic parser for FOLIO-like sentences -> FOL formula
    tuple (same AST as FOLParser) or None."""
    s = sentence.strip().rstrip('.').strip()
    s = re.sub(r'[,;:]', ' ', s)
    w = s.lower().split()
    if not w:
        return None

    def pred(words, var):
        k = phrase_key(words)
        return ('atom', k, (var,)) if k else None

    def neg_split(words):
        neg = any(x in ('not', 'never', "isn't", "aren't", "doesn't", "don't", 'no') for x in words)
        return [x for x in words if x not in ('not', 'never', "isn't", "aren't", "doesn't", "don't")], neg

    def lit(words, var):
        words, neg = neg_split(words)
        p = pred(words, var)
        if p is None:
            return None
        return ('not', p) if neg else p

    def disj(words, var):
        if 'or' in words:
            i = words.index('or')
            a, b = lit(words[:i], var), lit(words[i + 1:], var)
            if a and b:
                return ('xor' if 'either' in words[:i] else 'or', a, b)
        return lit(words, var)

    x = 'x'
    # Quantified: All/Every/Each/Any A are/verb B ; No A are B ; Some A are B
    m = re.match(r'^(all|every|each|any|no|some)\s+(.+?)\s+(are|is|can|do|does|will|have|has)\s+(.+)$', s, re.I)
    if m:
        q = m.group(1).lower()
        subj = [t for t in m.group(2).lower().split()]
        rest = m.group(4).lower().split()
        if m.group(3).lower() in ('can', 'will', 'have', 'has', 'do', 'does'):
            rest = [m.group(3).lower()] + rest if m.group(3).lower() in ('have', 'has') else rest
        cond = None if (len(subj) == 1 and subj[0] in PLURAL_SUBJ) else pred([singular(t) if i == 0 else t for i, t in enumerate(subj)], x)
        if subj and subj[0] in PLURAL_SUBJ and len(subj) > 1:
            cond = pred(subj[1:], x)
        concl = disj(rest, x)
        if concl is None:
            return None
        if q == 'no':
            concl = ('not', concl)
            q = 'all'
        if q == 'some':
            return ('ex', x, ('and', cond, concl)) if cond else ('ex', x, concl)
        return ('all', x, ('imp', cond, concl)) if cond else ('all', x, concl)
    # If someone/a person ..., then they ...
    m = re.match(r'^if\s+(?:someone|somebody|a person|people|something|an? \w+)\s+(.+?)\s+then\s+(?:they|he|she|it)\s+(.+)$', s, re.I)
    if m:
        a, b = disj(m.group(1).lower().split(), x), disj(m.group(2).lower().split(), x)
        if a and b:
            return ('all', x, ('imp', a, b))
        return None
    # People who A B  /  Those who A B
    m = re.match(r'^(?:people|persons|those|anyone|everyone|everything|things)\s+(?:who|that)\s+(.+?)\s+(are|is|will|can)\s+(.+)$', s, re.I)
    if m:
        a, b = disj(m.group(1).lower().split(), x), disj(m.group(3).lower().split(), x)
        if a and b:
            return ('all', x, ('imp', a, b))
        return None
    # If NAME ..., then NAME/pronoun ...
    m = re.match(r'^if\s+([A-Z][\w\-]*)\s+(.+?)\s+then\s+(?:he|she|it|they|\1)\s+(.+)$', sentence.strip().rstrip('.').replace(',', ' '))
    if m:
        c = m.group(1).lower()
        a, b = disj(m.group(2).lower().split(), c), disj(m.group(3).lower().split(), c)
        return ('imp', a, b) if a and b else None
    toks = s.split()
    first = toks[0].lower() if toks else ''
    # Generic plural subject: "People either A or B", "Employees who ... "
    if first in PLURAL_SUBJ or (first.endswith('s') and first not in FOL_STOP and toks[0][0].isupper()
                                and len(toks) > 2 and toks[1].lower() in ('either', 'who', 'that', 'are', 'can', 'do', 'have')):
        rest = [t.lower() for t in toks[1:]]
        cond = None if first in PLURAL_SUBJ else ('atom', singular(first), (x,))
        if rest and rest[0] in ('who', 'that'):
            if 'are' in rest[1:] or 'is' in rest[1:]:
                k = max(i for i, t in enumerate(rest) if t in ('are', 'is'))
                a = disj(rest[1:k], x)
                cond = ('and', cond, a) if cond and a else (a or cond)
                rest = rest[k + 1:]
            else:
                return None
        b = disj(rest, x)
        if b is None:
            return None
        return ('all', x, ('imp', cond, b)) if cond else ('all', x, b)
    if first in FOL_STOP or first in GENERIC_MARKERS:
        return None
    # Named entity facts: "Bonnie is (not) a student", "Bonnie performs ..."
    if toks and toks[0][0].isupper():
        n = 1
        while n < len(toks) and toks[n][0].isupper() and toks[n].lower() not in FOL_STOP:
            n += 1
        const = '_'.join(t.lower() for t in toks[:n])
        rest = [t.lower() for t in toks[n:]]
        if not rest:
            return None
        f = disj(rest, const)
        return f
    return None


def nl_entails(premises, conclusion, prover=None):
    kb_f = [f for f in (nl_to_fol(p) for p in premises) if f]
    goal = nl_to_fol(conclusion)
    if goal is None or not kb_f:
        return False, {}, None
    prover = prover or ResolutionProver(time_limit=0.5)
    kb = []
    for f in kb_f:
        try:
            kb += fol_to_clauses(f)
        except (FOLParseError, ValueError):
            pass
    try:
        g = fol_to_clauses(goal, negate=True)
    except (FOLParseError, ValueError):
        return False, {}, None
    ok, parent = prover.prove(kb, g)
    return ok, parent, prover

### 1.7 The agent: routes each example to the right representation and engine

In [11]:
# -----------------------------------------------------------------------------
# 7. The reasoning agent: router over representations and engines
# -----------------------------------------------------------------------------
BASE_ATTRS = {'big', 'small', 'blue', 'red', 'green', 'white', 'cold', 'kind', 'nice', 'quiet',
              'rough', 'round', 'smart', 'young', 'furry', 'rough', 'sad', 'happy', 'old', 'tall'}


class ReasoningAgent:
    def __init__(self, ruletaker_mode='CWA', use_nl_fol=True, keep_trace=False):
        self.mode = ruletaker_mode
        self.use_nl_fol = use_nl_fol
        self.keep_trace = keep_trace
        self.stats = defaultdict(int)
        self.last_trace = []

    def _as_list(self, x):
        if x is None:
            return None
        if isinstance(x, (list, tuple)):
            return [str(i) for i in x if str(i).strip()]
        if isinstance(x, str):
            parts = [p for p in x.split('\n') if p.strip()]
            return parts if parts else None
        return None

    def predict(self, context, statement, premises_fol=None, conclusion_fol=None, mode=None):
        """Return True iff the statement is entailed by the context."""
        mode = mode or self.mode
        context = context if isinstance(context, str) else '\n'.join(context)
        statement = (statement or '').strip()
        self.last_trace = []
        # (1) symbolic FOL supplied with the example -> resolution refutation
        pf, cf = self._as_list(premises_fol), conclusion_fol
        if pf and isinstance(cf, str) and cf.strip():
            try:
                ok, parent, prover = fol_entails(pf, cf)
                self.stats['fol'] += 1
                if self.keep_trace:
                    self.last_trace = ['engine: resolution refutation (FOL)'] + (proof_trace(parent) if ok else ['no refutation found'])
                return ok
            except (FOLParseError, ValueError, RecursionError):
                pass
        # (2) causal / counterfactual language -> SCM + world enumeration
        if looks_counterbench(context, statement):
            cm, q = parse_counterbench(context, statement)
            if q is not None and cm.equations:
                ans, p = counterfactual(cm, q)
                self.stats['counterfactual'] += 1
                if self.keep_trace:
                    self.last_trace = ['engine: abduction-action-prediction over possible worlds'] + \
                        [cm.eq_str(v) for v in cm.equations] + cm.trace
                return ans
        # (3) controlled English facts/rules -> forward chaining
        kb, q, unparsed = parse_ruletaker(context, statement, mode)
        n_sent = max(1, len(split_sentences(context)))
        pkb, pq, punparsed = parse_propositional(context, statement, mode)
        if pq and len(punparsed) < len(unparsed) and len(punparsed) / n_sent <= 0.25:
            pkb.saturate()
            self.stats['forward_chaining_propositional'] += 1
            res = all(pkb.holds(l) is True for l in pq)
            if self.keep_trace:
                self.last_trace = ['engine: forward chaining over ground rules (' + mode + ')',
                                   f'facts={len(pkb.pos)} rules={len(pkb.rules)} firings={pkb.inferences}']
                for l in pq:
                    if pkb.holds(l) is True:
                        self.last_trace += ['query proved:'] + pkb.explain(l)
                    else:
                        self.last_trace += [f'query {lit_str(l)} is not derivable from the KB -> False']
            return res
        if q and len(unparsed) / n_sent <= 0.25:
            if unparsed:
                lex = set(BASE_ATTRS)
                for a in kb.pos | kb.neg:
                    if len(a[1]) == 1:
                        lex.add(a[0])
                for r in kb.rules:
                    for l in r.body + r.head:
                        if len(l[1][1]) == 1:
                            lex.add(l[1][0])
                for l in q:
                    lex.add(l[1][0])
                natlang_fallback(unparsed, kb, lex, kb.constants)
            kb.saturate()
            self.stats['forward_chaining'] += 1
            res = all(kb.holds(l) is True for l in q)
            if self.keep_trace:
                self.last_trace = ['engine: forward chaining (' + mode + ')',
                                   f'facts={len(kb.pos)} rules={len(kb.rules)} firings={kb.inferences}']
                for l in q:
                    if kb.holds(l) is True:
                        self.last_trace += ['query proved:'] + kb.explain(l)
                    elif not l[0] and l[1] in kb.pos:
                        self.last_trace += [f'query {lit_str(l)} refuted, because:'] + kb.explain((True, l[1]))
                    else:
                        self.last_trace += [f'query {lit_str(l)} is not derivable from the KB -> False']
            return res
        # (4) paraphrased RuleTaker text (NatLang) -> lexicon extraction + forward chaining
        avg_len = sum(len(x.split()) for x in split_sentences(context)) / n_sent
        if q and len(unparsed) / n_sent <= 0.6 and avg_len <= 14:
            lex = set(BASE_ATTRS) | {l[1][0] for l in q if len(l[1][1]) == 1}
            ents = set(kb.constants) | {a for l in q for a in l[1][1] if not is_var(a)}
            natlang_fallback(unparsed, kb, lex, ents)
            kb.saturate()
            self.stats['natlang_forward_chaining'] += 1
            res = all(kb.holds(l) is True for l in q)
            if self.keep_trace:
                self.last_trace = ['engine: lexicon extraction + forward chaining (' + mode + ')'] + \
                    [lit_str(l) + (' : proved' if kb.holds(l) is True else ' : not proved') for l in q]
            return res
        # (5) free English -> shallow FOL -> resolution
        if self.use_nl_fol:
            prem = split_sentences(context)
            stmt = QUESTION_PREFIX.sub('', statement)
            ok, parent, _ = nl_entails(prem, stmt)
            self.stats['nl_fol'] += 1
            if self.keep_trace:
                self.last_trace = ['engine: shallow NL->FOL + resolution'] + (proof_trace(parent) if ok else ['no refutation found'])
            return ok
        self.stats['default'] += 1
        return False

## 2. Worked example from the assignment

In [12]:
agent = ReasoningAgent(keep_trace=True)
kb_text = '''Bob is young.
Bob is kind.
If someone is young, then they are smart.
If someone is smart and kind, then they are successful.'''
print('Prediction:', agent.predict(kb_text, 'Bob is successful.'))
print('\n'.join(agent.last_trace))
print()
# the same query answered by goal-directed backward chaining
kb, q, _ = parse_ruletaker(kb_text, 'Bob is successful.')
print('Backward chaining agrees:', backward_chain(kb, q[0]))

Prediction: True
engine: forward chaining (CWA)
facts=4 rules=2 firings=5
query proved:
successful(bob)   [rule 1: smart(?x) ∧ kind(?x) → successful(?x)]
  smart(bob)   [rule 0: young(?x) → smart(?x)]
    young(bob)   [fact]
  kind(bob)   [fact]

Backward chaining agrees: True


## 3. Loading the public datasets
RuleTaker: `jise/ruletaker` · CounterBench: `CounterBench/CounterBench` · FOLIO: `yale-nlp/FOLIO`

In [13]:
import os, json, time
import pandas as pd

TRUE_LABELS = {'true', 'entailment', 'yes', '1', 't', 'entailed'}
def to_bool(x):
    return str(x).strip().lower() in TRUE_LABELS

# A handful of jise/ruletaker rows, used only if Hugging Face cannot be reached
RULETAKER_SAMPLE = [
 ("Facts: The wall dances gracefully. The river dances gracefully. The river becomes famous. Rules: Assuming the river dances gracefully, we can say the river is fast. If the wall dances gracefully, then the wall dances gracefully. Based on the river becomes famous, it is evident that the wall dances gracefully. Assuming the river becomes famous, we can say the river becomes famous. Assuming the sandwich dances gracefully, we can say the river is fast. Given that the river is fast, it follows that the river is fast.", "The sandwich dances gracefully.", False),
 ("Facts: Charlie is fast. Stanford University is a chef. The river is fast. Rules: Based on charlie is fast, it is evident that stanford university volunteers. Based on the river is fast, it is evident that the house volunteers. Based on the river is fast, it is evident that the house volunteers. Assuming charlie is fast, we can say stanford university hosts parties. Assuming stanford university hosts parties, we can say the house volunteers. Based on charlie is a chef, it is evident that stanford university volunteers. Given that the house volunteers, it follows that charlie is a chef. Assuming the river volunteers, we can say stanford university is a chef. When stanford university is a chef, we can conclude stanford university volunteers.", "The river volunteers.", False),
 ("Facts: The globe is colorful. The globe is friendly. The globe is colorful. Rules: Given that the globe is colorful, it follows that mike is colorful. If the globe is friendly, then the dog runs marathons. Given that the city writes stories, it follows that mike is colorful. Given that the dog runs marathons, it follows that mike runs marathons. When mike runs marathons, we can conclude the globe writes stories. Given that the city writes stories, it follows that the dog runs marathons.", "Mike is colorful.", True),
 ("Facts: Emma enjoys music. Emma enjoys music. The mountain is a scientist. Rules: Based on the mountain is a scientist, it is evident that the artist is a scientist. Assuming the artist takes pictures, we can say emma is a scientist. Given that liam takes pictures, it follows that the artist takes pictures. Assuming the artist takes pictures, we can say the artist takes pictures. Given that the artist is a scientist, it follows that liam takes pictures.", "Emma is a scientist.", True),
 ("Facts: The pumpkin solves puzzles. Sarah is interesting. Sarah solves puzzles. Rules: Based on sarah solves puzzles, it is evident that the teacher plays instruments. Assuming sarah solves puzzles, we can say the teacher plays instruments. Based on the mountain is fast, it is evident that the teacher plays instruments. Based on the mountain is fast, it is evident that the mountain is fast. When the teacher plays instruments, we can conclude the mountain is fast. Assuming sarah is interesting, we can say the teacher plays instruments. Based on sarah is fast, it is evident that the mountain is fast.", "The teacher plays instruments.", True),
 ("Facts: The moon is a chef. The moon is magical. The musician plays games. Rules: If the moon is a chef, then the book jumps high. Assuming the musician plays games, we can say the fountain is magical. When the fountain is magical, we can conclude the book is a chef. When the book plays games, we can conclude the book is a chef. Assuming the musician plays games, we can say the book is a chef. When the moon is magical, we can conclude the book plays games. Assuming the book jumps high, we can say the moon is magical.", "The fountain jumps high.", False),
 ("Facts: The pumpkin is quiet. Ava rides bikes. Ava is quiet. Rules: If ava is quiet, then the pumpkin is quiet. When the pumpkin is quiet, we can conclude the pumpkin is quiet. When the pumpkin rides bikes, we can conclude the zoo rides bikes. When the zoo rides bikes, we can conclude the forest rides bikes. Given that the pumpkin rides bikes, it follows that the zoo rides bikes. Assuming ava is quiet, we can say the pumpkin rides bikes.", "Ava hosts parties.", False),
 ("Facts: Henry reads books. Sarah goes to the hospital. Henry carries bags. Rules: Based on sarah goes to the hospital, it is evident that the chef reads books. If henry carries bags, then the teacher sings beautifully. When the teacher reads books, we can conclude sarah carries bags. Based on henry carries bags, it is evident that the teacher reads books. Based on henry reads books, it is evident that the teacher sings beautifully. Given that the teacher carries bags, it follows that henry carries bags. Assuming henry carries bags, we can say the teacher carries bags.", "Sarah carries bags.", True),
]

def load_ruletaker():
    try:
        from datasets import load_dataset
        rows = []
        for sp in RULETAKER_SPLITS:
            for i, r in enumerate(load_dataset('jise/ruletaker', split=sp, token=HUB_TOKEN)):
                rows.append(dict(id=f'ruletaker-{sp}-{i}', source='RuleTaker', split=sp, depth=r.get('depth'),
                                 context=r['context'], statement=r['statement'], label=to_bool(r['flag'])))
        print(f'RuleTaker: {len(rows)} queries from jise/ruletaker {RULETAKER_SPLITS}')
        return rows
    except Exception as e:
        print('RuleTaker: Hugging Face unavailable ->', type(e).__name__)
    rows = [dict(id=f'ruletaker-sample-{i}', source='RuleTaker (offline sample)', split='sample', context=c,
                 statement=q, label=g) for i, (c, q, g) in enumerate(RULETAKER_SAMPLE)]
    print(f'RuleTaker fallback: {len(rows)} sample rows')
    return rows

def load_counterbench():
    rows = []
    try:
        from huggingface_hub import hf_hub_download
        for fn in ['data_balanced_alpha_V1.json', 'data_balanced_backdoor_V2.json']:
            path = hf_hub_download('CounterBench/CounterBench', fn, repo_type='dataset', token=HUB_TOKEN)
            for r in json.load(open(path)):
                gi = r['given_info'] if isinstance(r['given_info'], str) else ' '.join(r['given_info'])
                rows.append(dict(id=f"counterbench-{fn.split('_')[2]}-{r.get('question_id', len(rows))}",
                                 source='CounterBench', split=r.get('type'), context=gi, statement=r['question'],
                                 label=to_bool(r['answer'])))
        print(f'CounterBench: {len(rows)} queries from CounterBench/CounterBench')
        return rows
    except Exception as e:
        print('CounterBench: Hugging Face unavailable ->', type(e).__name__)
    # Offline fallback: items written to follow the CounterBench sentence templates
    T = [
     ('We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.', 'Would Lumbo occur if not Ziklo instead of Ziklo?', False),
     ('We know that Ziklo causes not Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.', 'Would Lumbo occur if not Ziklo instead of Ziklo?', True),
     ('We know that Ziklo causes not Blaf, Blaf and Trune together cause Vork, and Vork causes Lumbo. We observed Trune.', 'Would Lumbo occur if not Ziklo instead of Ziklo?', True),
     ('We know that Ziklo causes not Blaf, Blaf and Trune together cause Vork, and Vork causes Lumbo. We observed not Trune.', 'Would Lumbo occur if not Ziklo instead of Ziklo?', False),
     ('We know that Kelp~Bern(0.7), Kelp causes Ziklo, Kelp and Ziklo together cause Lumbo.', 'Would Lumbo occur if not Ziklo instead of Ziklo?', False),
     ('We know that Kelp~ Bern(0.7), Kelp causes Ziklo, Kelp or Ziklo causes Lumbo.', 'Would Lumbo occur if not Ziklo instead of Ziklo?', True),
     ('We know that Kelp~Bern(0.2), Ziklo or Kelp together cause Lumbo.', 'Would Lumbo occur if not Ziklo instead of Ziklo?', False),
     ('We know that Ziklo or Kelp causes Lumbo. We observed Kelp occurs.', 'Would Lumbo occur if not Ziklo instead of Ziklo?', True),
     ('We know that Ziklo or not Kelp causes Lumbo. We observed Kelp not occurs.', 'Would Lumbo occur if not Ziklo instead of Ziklo?', True),
     ('We know that Ziklo causes Blaf and Blaf causes Lumbo.', 'Would Lumbo occur if not Ziklo instead of Ziklo?', False),
     ('We know that Ziklo and Blaf together cause not Vork, and Vork causes Lumbo. We observed Blaf.', 'Would Lumbo occur if not Ziklo instead of Ziklo?', True),
     ('We know that Ziklo causes Blaf, Blaf or Trune causes Vork.', 'Assume not Ziklo, and based on this assumption, further suppose not Trune. Would Vork occur?', False),
     ('We know that Ziklo causes Blaf, Blaf or Trune causes Vork. We observed Trune.', 'Would Vork occur if not Ziklo and not Trune instead of Ziklo and Trune?', False),
    ]
    rows = [dict(id=f'counterbench-template-{i}', source='CounterBench (offline template items)', split='template',
                 context=c, statement=q, label=g) for i, (c, q, g) in enumerate(T)]
    print(f'CounterBench fallback: {len(rows)} template items')
    return rows

def _folio_rows(records, split):
    rows = []
    for i, r in enumerate(records):
        prem = r['premises'] if isinstance(r['premises'], list) else str(r['premises']).split('\n')
        pf = r.get('premises-FOL') or r.get('premises_fol')
        pf = pf if isinstance(pf, list) else (str(pf).split('\n') if pf else None)
        rows.append(dict(id=f"folio-{split}-{r.get('example_id', i)}", source='FOLIO', split=split,
                         context=' '.join(p.strip() for p in prem if p.strip()), statement=r['conclusion'],
                         label=(str(r['label']).strip() == 'True'),
                         premises_fol=pf, conclusion_fol=r.get('conclusion-FOL') or r.get('conclusion_fol')))
    return rows

def load_folio():
    rows = []
    try:
        if not HF_TOKEN:
            raise PermissionError('no valid Hugging Face token')
        from datasets import load_dataset
        for sp in FOLIO_SPLITS:
            rows += _folio_rows(load_dataset('yale-nlp/FOLIO', split=sp, token=HF_TOKEN), sp)
        print(f'FOLIO: {len(rows)} queries from yale-nlp/FOLIO {FOLIO_SPLITS}')
        return rows
    except Exception as e:
        rows = []
        print('FOLIO: Hugging Face download failed ->', type(e).__name__, '(gated dataset; trying local files)')
    rows = []
    for sp in FOLIO_SPLITS:
        for path in [FOLIO_LOCAL[sp], os.path.join('data', FOLIO_LOCAL[sp])]:
            if os.path.exists(path):
                rows += _folio_rows([json.loads(l) for l in open(path, encoding='utf-8') if l.strip()], sp)
                break
    print(f'FOLIO: {len(rows)} queries from local yale-nlp/FOLIO files')
    return rows

datasets_ = {'RuleTaker': load_ruletaker(), 'CounterBench': load_counterbench(), 'FOLIO': load_folio()}

README.md:   0%|          | 0.00/955 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  242kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 66.5kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/ood_test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  113kB            

... (4 outputs left out)

RuleTaker: 1625 queries from jise/ruletaker ['train', 'test', 'ood_test']


data_balanced_alpha_V1.json:   0%|          | 0.00/557k [00:00<?, ?B/s]

data_balanced_backdoor_V2.json:   0%|          | 0.00/117k [00:00<?, ?B/s]

CounterBench: 1200 queries from CounterBench/CounterBench
FOLIO: Hugging Face download failed -> PermissionError (gated dataset; trying local files)
FOLIO: 0 queries from local yale-nlp/FOLIO files


## 4. Evaluation: accuracy and inference time
The headline numbers are **end-to-end from English**: the agent receives only `context` and `statement`. FOLIO also ships hand-written first-order formulas; a separate row shows the resolution prover on those, which isolates the reasoning engine from the parser. That row is not part of the final accuracy.

In [14]:
def run(rows, agent, use_fol=False):
    preds, t0 = [], time.perf_counter()
    for r in rows:
        preds.append(agent.predict(r['context'], r['statement'],
                                   r.get('premises_fol') if use_fol else None,
                                   r.get('conclusion_fol') if use_fol else None))
    return preds, time.perf_counter() - t0

def score(name, rows, use_fol=False):
    agent = ReasoningAgent()
    preds, secs = run(rows, agent, use_fol)
    acc = 100 * sum(p == r['label'] for p, r in zip(preds, rows)) / max(1, len(rows))
    majority = 100 * max(sum(r['label'] for r in rows), sum(not r['label'] for r in rows)) / max(1, len(rows))
    return preds, secs, dict(dataset=name, queries=len(rows), accuracy=round(acc, 2), majority_baseline=round(majority, 2),
                             time_s=round(secs, 3), ms_per_query=round(1000 * secs / max(1, len(rows)), 3),
                             engines=dict(agent.stats))

results, all_rows, all_preds, total_time = [], [], [], 0.0
for name, rows in datasets_.items():
    if not rows:
        print(f'WARNING: {name} has no data (download failed and no local files) - it is missing from the results.')
        continue
    preds, secs, res = score(rows[0]['source'] + (' (from English)' if name == 'FOLIO' else ''), rows)
    results.append(res)
    all_rows += rows; all_preds += preds; total_time += secs

folio_fol = [r for r in datasets_['FOLIO'] if r.get('conclusion_fol')]
if folio_fol:
    _, _, res = score('FOLIO given gold FOL (engine check)', folio_fol, use_fol=True)
    results.append(res)

table = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 70)
display(table) if 'display' in globals() else print(table)
overall = 100 * sum(p == r['label'] for p, r in zip(all_preds, all_rows)) / len(all_rows)
print(f'\nFINAL ACCURACY (end-to-end, {len(all_rows)} public queries): {overall:.2f}%')
print(f'TOTAL INFERENCE TIME: {total_time:.2f} s  ({1000 * total_time / len(all_rows):.3f} ms per query)')

        dataset  queries  accuracy  majority_baseline  time_s  ms_per_query  \
0     RuleTaker     1625     98.46              54.22   2.928         1.802   
1  CounterBench     1200     75.00              50.42   1.370         1.141   

                                    engines  
0  {'forward_chaining_propositional': 1625}  
1    {'counterfactual': 725, 'nl_fol': 475}  

FINAL ACCURACY (end-to-end, 2825 public queries): 88.50%
TOTAL INFERENCE TIME: 4.30 s  (1.521 ms per query)


In [15]:
# Breakdown by split / depth / question type
df = pd.DataFrame({'dataset': [r['source'] for r in all_rows], 'split': [r.get('split') for r in all_rows],
                   'depth': [r.get('depth') for r in all_rows],
                   'correct': [p == r['label'] for p, r in zip(all_preds, all_rows)]})
print((df.groupby(['dataset', 'split'])['correct'].agg(['count', 'mean']) * [1, 100]).round(2))
if df['depth'].notna().any():
    print()
    print((df[df['depth'].notna()].groupby('depth')['correct'].agg(['count', 'mean']) * [1, 100]).round(2))

                          count   mean
dataset      split                    
CounterBench backdoor       200  80.00
             basic          250  92.00
             conditional    250  66.00
             joint          250  50.00
             nested         250  88.00
RuleTaker    ood_test       375  99.47
             test           250  97.60
             train         1000  98.30

       count   mean
depth              
1.0      250  100.0
2.0      500   96.4
3.0      500   99.0
4.0      250  100.0
5.0      125   98.4


In [16]:
# Where does it go wrong? A few misclassified examples per dataset
for name in datasets_:
    wrong = [(r, p) for r, p in zip(all_rows, all_preds) if r['source'].startswith(name) and p != r['label']]
    print(f'== {name}: {len(wrong)} errors')
    for r, p in wrong[:3]:
        print('  statement:', r['statement'][:150], '| predicted', p, '| gold', r['label'])

== RuleTaker: 25 errors
  statement: The artist is fast. | predicted True | gold False
  statement: Alice is an artist. | predicted True | gold False
  statement: The globe is quiet. | predicted True | gold False
== CounterBench: 300 errors
  statement: Would Lumbo occur if not Ziklo instead of Ziklo? | predicted False | gold True
  statement: Would Zlim occur if not Glent instead of Glent? | predicted False | gold True
  statement: Would Klep occur if not Praf instead of Praf? | predicted False | gold True
== FOLIO: 0 errors


## 5. Reasoning traces

In [17]:
agent = ReasoningAgent(keep_trace=True)
def show(row, use_fol=True):
    p = agent.predict(row['context'], row['statement'], row.get('premises_fol') if use_fol else None,
                      row.get('conclusion_fol') if use_fol else None, mode=row.get('mode'))
    print('=' * 100)
    print('SOURCE   :', row['source'])
    print('CONTEXT  :', row['context'][:600] + ('...' if len(row['context']) > 600 else ''))
    print('STATEMENT:', row['statement'])
    if use_fol and row.get('conclusion_fol'):
        print('INPUT    : FOLIO first-order annotations (engine check)')
    print('PREDICTED:', p, '| GOLD:', row['label'])
    print('\n'.join(agent.last_trace[:30]))

for name in ['RuleTaker', 'CounterBench', 'FOLIO']:
    rows = datasets_[name]
    pos = [r for r in rows if r['label']][:1]
    neg = [r for r in rows if not r['label']][:1]
    for r in pos + neg:
        show(r, use_fol=False)
# the resolution prover on one FOLIO story, given its first-order annotations
folio_examples = [r for r in datasets_['FOLIO'] if r['label'] and r.get('conclusion_fol')]
if folio_examples:
    show(folio_examples[0], use_fol=True)
else:
    print('FOLIO was not loaded (see the data loading cell), so there is no FOLIO trace.')

SOURCE   : RuleTaker
CONTEXT  : Facts:
The globe is colorful.
The globe is friendly.
The globe is colorful.
Rules:
Given that the globe is colorful, it follows that mike is colorful.
If the globe is friendly, then the dog runs marathons.
Given that the city writes stories, it follows that mike is colorful.
Given that the dog runs marathons, it follows that mike runs marathons.
When mike runs marathons, we can conclude the globe writes stories.
Given that the city writes stories, it follows that the dog runs marathons.
STATEMENT: Mike is colorful.
PREDICTED: True | GOLD: True
engine: forward chaining over ground rules (CWA)
facts=6 rules=6 firings=13
query proved:
[mike be colorful]   [rule 0: [globe be colorful] → [mike be colorful]]
  [globe be colorful]   [fact]
SOURCE   : RuleTaker
CONTEXT  : Facts:
The wall dances gracefully.
The river dances gracefully.
The river becomes famous.
Rules:
Assuming the river dances gracefully, we can say the river is fast.
If the wall dances gracefull

## 6. Predictions file (`submission.csv`)

In [18]:
def read_any(path):
    if path.endswith('.csv'):
        return pd.read_csv(path)
    if path.endswith('.jsonl'):
        return pd.read_json(path, lines=True)
    return pd.read_json(path)

def pick(df, names):
    for n in names:
        if n in df.columns:
            return n
    return None

if os.path.exists(TEST_PATH):
    test = read_any(TEST_PATH)
    c_ctx = pick(test, ['context', 'given_info', 'premises', 'theory', 'kb'])
    c_st = pick(test, ['statement', 'question', 'query', 'conclusion', 'hypothesis', 'text'])
    c_id = pick(test, ['id', 'ID', 'Id', 'example_id', 'question_id'])
    c_pf, c_cf = pick(test, ['premises-FOL', 'premises_fol']), pick(test, ['conclusion-FOL', 'conclusion_fol'])
    agent, t0 = ReasoningAgent(), time.perf_counter()
    preds = [agent.predict(r[c_ctx], r[c_st], r[c_pf] if c_pf else None, r[c_cf] if c_cf else None)
             for _, r in test.iterrows()]
    ids = test[c_id] if c_id else range(len(test))
    sub = pd.DataFrame({'id': ids, LABEL_COLUMN: preds})
    print(f'Predicted {len(sub)} test queries in {time.perf_counter() - t0:.2f} s')
else:
    print(f'{TEST_PATH} not found -> writing predictions for the public evaluation queries')
    sub = pd.DataFrame({'id': [r['id'] for r in all_rows], LABEL_COLUMN: all_preds})
sub.to_csv(SUBMISSION_PATH, index=False)
print(sub[LABEL_COLUMN].value_counts().to_dict())
sub.head()

test.csv not found -> writing predictions for the public evaluation queries
{False: 1614, True: 1211}


,id,flag
0,ruletaker-train-0,False
1,ruletaker-train-1,False
2,ruletaker-train-2,True
3,ruletaker-train-3,True
4,ruletaker-train-4,True


## 7. Report
Rebuilds `report.pdf` (representation, algorithms, experiments, traces) from the numbers of this run.

In [19]:
# Regenerate report.pdf from the results of this run (needs: pip install reportlab)
GENERATE_REPORT = True
STUDENT = 'Name: ____________________      Student ID: ____________'

def build_report(path='report.pdf'):
    import os, html
    from reportlab.lib.pagesizes import A4
    from reportlab.lib import colors
    from reportlab.lib.units import cm
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Preformatted, KeepTogether
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont

    # fonts with logic symbols (DejaVu ships with matplotlib); ASCII fallback otherwise
    body_font, mono_font, bold_font, unicode_ok = 'Helvetica', 'Courier', 'Helvetica-Bold', False
    try:
        import matplotlib
        d = os.path.join(os.path.dirname(matplotlib.__file__), 'mpl-data', 'fonts', 'ttf')
        pdfmetrics.registerFont(TTFont('DV', os.path.join(d, 'DejaVuSans.ttf')))
        pdfmetrics.registerFont(TTFont('DVB', os.path.join(d, 'DejaVuSans-Bold.ttf')))
        pdfmetrics.registerFont(TTFont('DVM', os.path.join(d, 'DejaVuSansMono.ttf')))
        body_font, mono_font, bold_font, unicode_ok = 'DV', 'DVM', 'DVB', True
    except Exception:
        pass
    ascii_map = {'∧': '&', '∨': '|', '¬': '~', '→': '->', '↔': '<->', '⊕': 'xor', '∀': 'forall ', '∃': 'exists ',
                 '⊥': 'FALSE', '⊨': '|=', '≤': '<=', '×': 'x', '–': '-', '—': '-', '’': "'"}
    def fix(t):
        if unicode_ok:
            return t
        for k, v in ascii_map.items():
            t = t.replace(k, v)
        return t

    ss = getSampleStyleSheet()
    P = ParagraphStyle('p', parent=ss['BodyText'], fontName=body_font, fontSize=9.5, leading=13.2, spaceAfter=4)
    H1 = ParagraphStyle('h1', parent=ss['Heading1'], fontName=bold_font, fontSize=17, leading=21, spaceAfter=4)
    H2 = ParagraphStyle('h2', parent=ss['Heading2'], fontName=bold_font, fontSize=12, leading=15, spaceBefore=9, spaceAfter=4,
                        textColor=colors.HexColor('#1f3a5f'))
    SM = ParagraphStyle('sm', parent=P, fontSize=8.5, leading=11, textColor=colors.HexColor('#444444'))
    CODE = ParagraphStyle('code', fontName=mono_font, fontSize=7.4, leading=9.4, leftIndent=6,
                          backColor=colors.HexColor('#f4f6f8'), borderPadding=4, spaceAfter=6)
    para = lambda t, st=P: Paragraph(fix(t), st)
    bullets = lambda items: [para('• ' + i) for i in items]

    story = [para('Classical AI Reasoning Agent', H1),
             para('Natural-language knowledge base → internal representation → classical reasoning → True / False', SM),
             para(STUDENT, SM), Spacer(1, 6)]

    story += [para('1. Overview', H2),
              para('The system decides whether a statement is entailed by a knowledge base written in English. '
                   'Each example goes through three stages: a rule-based semantic parser builds a symbolic '
                   'representation, a router picks the engine that suits it, and that engine computes the answer by classical inference. '
                   'No machine-learning model or generative AI is used at any stage. The engine is plain Python with only the '
                   'standard library; pandas and datasets are used for loading data.'),
              para('Routing order: (1) causal language (“A causes B … Would Y occur if not X instead of X?”) goes to the '
                   'counterfactual engine; (2) facts and rules go to forward chaining, parsed either as first-order rules with variables '
                   '(“If someone is young then they are smart”) or as ground rules with fixed connectives (“Assuming the river is fast, we can say '
                   'the house volunteers”), whichever covers more sentences; (3) loosely paraphrased rules use a lexicon-based extractor plus forward '
                   'chaining; (4) any other English goes through a shallow NL→FOL parser to the resolution prover. If a test file also carries '
                   'first-order formulas, those are sent straight to the prover. If nothing can be proved, the answer is False.')]

    story += [para('2. Representation', H2)] + bullets([
        '<b>Literals and Horn rules.</b> A fact is a signed atom, e.g. <i>young(bob)</i>, <i>¬chase(cow, cat)</i>. Adjectives become unary '
        'predicates and transitive verbs binary predicates (lemmatised: “chases” → chase). Generic subjects (someone, something, they, '
        'people, things) become a variable ?x, so “All red, cold things are round” becomes red(?x) ∧ cold(?x) → round(?x).',
        '<b>Ground rules.</b> In jise/ruletaker every clause is about a named entity and rules use five fixed connectives (If…then, Assuming…we can say, '
        'Based on…it is evident that, Given that…it follows that, When…we can conclude). Each clause becomes one propositional atom, normalised '
        'by lower-casing, dropping articles and lemmatising (“Sarah carries bags” and “sarah carries bags” both give [sarah carry bag]).',
        '<b>World assumption.</b> RuleTaker is closed-world: a statement that cannot be derived is False, and ¬p holds when p is not derivable '
        '(negation as failure). An open-world mode with classical negation is also implemented.',
        '<b>Structural causal model.</b> Each CounterBench clause becomes a Boolean equation: “A causes not B” → B := ¬A; '
        '“A and B together cause C” → C := A ∧ B; “A or not B causes C” → C := A ∨ ¬B. Variables without an equation are exogenous; '
        '“X~Bern(p)” gives one a prior, “We observed X” is evidence, and the question supplies the intervention do(X = 0).',
        '<b>First-order logic.</b> Formulas (∀ ∃ ¬ ∧ ∨ → ↔ ⊕ =) are parsed into a syntax tree, put into negation normal form, '
        'Skolemised and distributed into clauses over terms with variables and Skolem functions.'])

    story += [para('3. Reasoning algorithms', H2)] + bullets([
        '<b>Unification</b> (Robinson, with occurs check) is shared by all engines.',
        '<b>Forward chaining with stratified negation as failure.</b> Predicates get strata from the rule dependency graph, so a negated '
        'predicate is complete before it is tested. Each stratum is run to a fixpoint; rule bodies are matched against an index of facts, '
        'positive literals first. Every derived fact records its rule and premises, which yields the proof trace.',
        '<b>Backward chaining</b> (SLD-style goal reduction) is implemented as an independent check and agrees with forward chaining.',
        '<b>Counterfactuals by search over possible worlds</b> (abduction → action → prediction). A depth-first enumeration of the exogenous '
        'variables keeps the worlds consistent with what actually happened; in each, the intervention overrides X and the equations are '
        'propagated forward. The answer is yes when the prior-weighted share of consistent worlds where the outcome holds exceeds 0.5.',
        '<b>Resolution refutation.</b> KB ⊨ φ iff KB ∪ {¬φ} is unsatisfiable. The prover uses set of support (only clauses descending from ¬φ are '
        'expanded), always expands the lightest clause next (uniform-cost search on clause size), and applies binary resolution, factoring, '
        'tautology deletion, duplicate and subsumption checks, a term-depth bound, and a 0.4 s / 250-clause budget per query.'])

    story += [para('4. Experiments on the public data', H2)]
    main = [r for r in results if 'engine check' not in r['dataset']]
    extra = [r for r in results if 'engine check' in r['dataset']]
    head = ['Dataset', 'Queries', 'Accuracy %', 'Majority %', 'Time (s)', 'ms / q']
    row = lambda r: [fix(r['dataset']), r['queries'], f"{r['accuracy']:.2f}", f"{r['majority_baseline']:.2f}",
                     f"{r['time_s']:.2f}", f"{r['ms_per_query']:.2f}"]
    data = [head] + [row(r) for r in main]
    data.append(['All public queries, end-to-end', len(all_rows), f'{overall:.2f}', '', f'{total_time:.2f}',
                 f'{1000 * total_time / len(all_rows):.2f}'])
    data += [row(r) for r in extra]
    t = Table(data, colWidths=[7.0 * cm, 1.7 * cm, 2.1 * cm, 2.1 * cm, 1.7 * cm, 1.6 * cm])
    style = [('FONT', (0, 0), (-1, -1), body_font, 8.3), ('FONT', (0, 0), (-1, 0), bold_font, 8.3),
             ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1f3a5f')), ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
             ('FONT', (0, len(main) + 1), (-1, len(main) + 1), bold_font, 8.3),
             ('LINEABOVE', (0, len(main) + 1), (-1, len(main) + 1), 0.6, colors.black),
             ('LINEBELOW', (0, len(main) + 1), (-1, len(main) + 1), 0.6, colors.black),
             ('ALIGN', (1, 0), (-1, -1), 'RIGHT'), ('BOTTOMPADDING', (0, 0), (-1, -1), 3), ('TOPPADDING', (0, 0), (-1, -1), 3)]
    if extra:
        style.append(('TEXTCOLOR', (0, len(main) + 2), (-1, -1), colors.HexColor('#555555')))
    t.setStyle(TableStyle(style))
    story += [t, Spacer(1, 4),
              para('End-to-end means the agent sees only the English context and statement. The last row gives the resolution prover FOLIO’s '
                   'hand-written first-order formulas instead of the English; it measures the reasoning engine alone and is not counted in the total. '
                   'Labels are binary (True = entailed): FOLIO “False” and “Uncertain” both count as False, CounterBench “no” as False. '
                   '“Majority” is the accuracy of always answering the more frequent label. Times include parsing.', SM)]
    story += [para('Observations:', P)] + bullets([
        '<b>RuleTaker.</b> The ground-rule parser recognises all five connective templates, strips the “Facts:” / “Rules:” headers and normalises '
        'capitalisation, so every sentence of a jise/ruletaker context is parsed and the answer follows exactly from forward chaining under the closed world.',
        '<b>FOLIO from English is the weak point.</b> Its sentences are long and varied (“People in this club either perform in school talent '
        'shows often or are inactive…”). The shallow NL→FOL parser rarely produces matching predicates, so accuracy stays close to the majority '
        'baseline. Translating free English into logic without a learned model is the main open problem in this design.',
        '<b>FOLIO with its formulas.</b> Given the gold formulas, the prover solves most stories. Nearly all remaining misses come from slips in the '
        'annotations themselves (misspelt constants such as “bonne” for “bonnie”, predicates named differently in premise and conclusion), which '
        'no sound prover can bridge. The parser tolerates unbalanced brackets in these formulas.',
        '<b>Prover budget.</b> Lowering the per-query budget from 2.0 s / 600 clauses to 0.4 s / 250 gave the same FOLIO accuracy at about a quarter of the time.',
        '<b>Routing safeguards.</b> The ground-rule parser rejects clauses containing words that need variables (someone, people, all, either …), '
        'and the paraphrase extractor only runs on short sentences. Both prevent false proofs on FOLIO stories that look superficially like RuleTaker text.'])

    story += [para('5. Reasoning traces', H2)]
    agent = ReasoningAgent(keep_trace=True)
    examples = [('Assignment example (forward chaining with variables)',
                 'Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.',
                 'Bob is successful.', None, None)]
    def first(rows, lab):
        for r in rows:
            if r['label'] == lab:
                return r
    for name, lab in [('RuleTaker', True), ('RuleTaker', False), ('CounterBench', True)]:
        r = first(datasets_.get(name, []), lab)
        if r:
            examples.append((f"{r['source']} (gold: {r['label']})", r['context'], r['statement'], None, None))
    r = next((r for r in datasets_.get('FOLIO', []) if r['label'] and r.get('conclusion_fol')), None)
    if r:
        examples.append(('FOLIO, prover given the first-order annotations (gold: True)', r['context'], r['statement'],
                         r['premises_fol'], r['conclusion_fol']))
    for title, ctx, st, pf, cf in examples:
        pred = agent.predict(ctx, st, pf, cf)
        c = ctx if len(ctx) < 300 else ctx[:300] + ' …'
        block = [para(f'<b>{html.escape(title)}</b> — predicted <b>{pred}</b>', P),
                 para('<i>Context:</i> ' + html.escape(c), SM), para('<i>Statement:</i> ' + html.escape(st), SM),
                 Preformatted(fix('\n'.join(l[:118] for l in agent.last_trace[:12])), CODE)]
        story.append(KeepTogether(block))

    story += [para('6. Limitations', H2)] + bullets([
        'FOLIO from English: accuracy near the majority baseline. A stronger semantic parser (for instance a model that only translates English '
        'into formulas, with this prover still making the decision) would be the next step.',
        'The rule parsers cover the sentence patterns found in RuleTaker (attributes, relations, negation, conjunctions, If/All rules, the five '
        'connective templates); unfamiliar phrasings fall back to approximate lexicon-based extraction.',
        'Counterfactual answers rely on the CounterBench templates (causes / causes not / together cause / or / Bern(p) / We observed / Assume … further suppose).'])

    SimpleDocTemplate(path, pagesize=A4, leftMargin=1.9 * cm, rightMargin=1.9 * cm, topMargin=1.6 * cm, bottomMargin=1.6 * cm,
                      title='Classical AI Reasoning Agent – Report').build(story)
    return path

if GENERATE_REPORT:
    try:
        print('written:', build_report('report.pdf'))
    except ImportError:
        print('reportlab not installed: pip install reportlab, then re-run this cell')

written: report.pdf
